<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_05_window_scaling_seq2one/stage_05_window_scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_05_window_scaling**




## **0. Configuración del Entorno**


### 0.1. Acceso a Drive

In [40]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [41]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [42]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

# ----------------------------
# Logging
# ----------------------------
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_06_window_scaling_seq2seq")

### 0.4. Definición de rutas

In [43]:
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

#### RUTAS DE ENTRADA

In [44]:
# ============================================================
# ENTRADA: T2 SPLITS
# ============================================================
IN_SUMMARY = DRIVE_DIR /  Path(os.environ.get("IN_PARQUET", "data/04_features/mnq_t2_summary.json"))
IN_SPLITS_SUMMARY = DRIVE_DIR / Path(os.environ.get("IN_SUMMARY", "data/05_splits/splits_summary.json"))
IN_PARQUET_T2_TRAIN = DRIVE_DIR / Path(os.environ.get("IN_PARQUET_T2_TRAIN", "data/05_splits/mnq_t2_train.parquet"))
IN_PARQUET_T2_VALID =  DRIVE_DIR / Path(os.environ.get("IN_PARQUET_T2_VALID", "data/05_splits/mnq_t2_valid.parquet"))
IN_PARQUET_T2_TEST =  DRIVE_DIR / Path(os.environ.get("IN_PARQUET_T2_TEST", "data/05_splits/mnq_t2_test.parquet"))




#### RUTAS DE SALIDA DE PARQUETS ESCALADOS

In [45]:
# ============================================================
# DATASETS ESCALADOS
# ============================================================
OUT_PARQUET_T2_TRAIN_Z = DRIVE_DIR / Path(os.environ.get("OUT_PARQUET_T2_TRAIN_Z", "data/06_scaled/mnq_t2_train_z.parquet"))
OUT_PARQUET_T2_VALID_Z =  DRIVE_DIR / Path(os.environ.get("OUT_PARQUET_T2_VALID_Z", "data/06_scaled/mnq_t2_valid_z.parquet"))
OUT_PARQUET_T2_TEST_Z  =  DRIVE_DIR / Path(os.environ.get("OUT_PARQUET_T2_TEST_Z",  "data/06_scaled/mnq_t2_test_z.parquet"))
OUT_SCALER_T2          =  DRIVE_DIR / Path(os.environ.get("OUT_SCALER_T2", "data/06_scaled/scaler_t2.pkl"))
OUT_SCALER_META_T2          =  DRIVE_DIR / Path(os.environ.get("OUT_SCALER_META_T2", "data/06_scaled/scaler_meta_t2.json"))

#### RUTAS DE SALIDA DE VENTANAS

In [46]:
from pathlib import Path
import os

# =========================
# SEQ2ONE (NPZ unificado por split)
# =========================

OUT_SEQ2ONE_T2_90_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_90_TRAIN", "data/07_windows/seq2one/windows_t2_90_train.npz")
OUT_SEQ2ONE_T2_90_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_90_VALID", "data/07_windows/seq2one/windows_t2_90_valid.npz")
OUT_SEQ2ONE_T2_90_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_90_TEST",  "data/07_windows/seq2one/windows_t2_90_test.npz")

OUT_SEQ2ONE_T2_120_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_120_TRAIN", "data/07_windows/seq2one/windows_t2_120_train.npz")
OUT_SEQ2ONE_T2_120_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_120_VALID", "data/07_windows/seq2one/windows_t2_120_valid.npz")
OUT_SEQ2ONE_T2_120_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_T2_120_TEST",  "data/07_windows/seq2one/windows_t2_120_test.npz")



# **1. Carga de datos**

## 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [47]:
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def load_mnq_parquet(path: Path):

    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    print("Archivo encontrado en disco. Cargando dataset local...")

    mnq_parquet = pd.read_parquet(path)

    # ==========================================================
    # Orden temporal obligatorio
    # ==========================================================
    mnq_parquet = mnq_parquet.sort_values(
        by=["date", "minute_of_day"],
        ascending=[True, True]
    )

    # ==========================================================
    # Verificación
    # ==========================================================
    ordenado = mnq_parquet[["date", "minute_of_day"]].reset_index(drop=True).equals(
        mnq_parquet[["date", "minute_of_day"]]
        .sort_values(["date", "minute_of_day"])
        .reset_index(drop=True)
    )

    print("Orden temporal correcto:", ordenado)

    return mnq_parquet

## **1.2. Información de datasets**


In [48]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")


## **1.3. Carga de mnq e información**


In [49]:
# =========================
# T2
# =========================
mnq_t2_train = load_mnq_parquet(IN_PARQUET_T2_TRAIN)
info_mnq_t2_train = mnq_dataset_info(mnq_t2_train, name="mnq_t2_train", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_t2_train)

mnq_t2_valid = load_mnq_parquet(IN_PARQUET_T2_VALID)
info_mnq_t2_valid = mnq_dataset_info(mnq_t2_valid, name="mnq_t2_valid", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_t2_valid)

mnq_t2_test = load_mnq_parquet(IN_PARQUET_T2_TEST)
info_mnq_t2_test = mnq_dataset_info(mnq_t2_test, name="mnq_t2_test", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_t2_test)


Archivo encontrado en disco. Cargando dataset local...
Orden temporal correcto: True
Dataset: mnq_t2_train
Shape: (62220, 12)
Columns: ['date', 'minute_of_day', 'ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd', 't2_p40_h30', 't2_p40_h60', 't2_p50_h30']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 09:30:00-05:00  ->  2024-05-13 10:29:00-04:00
First/Last day: 2020-01-02  ->  2024-05-13
Total days (trading): 1037
Time-of-day range (minutes): {'min_minute_of_day': 570, 'max_minute_of_day': 629}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 14:30:00+00:00  ->  2024-05-13 14:29:00+00:00
Archivo encontrado en disco. Cargando dataset local...
Orden temporal correcto: True
Dataset: mnq_t2_valid
Shape: (13320, 12)
Columns: ['date', 'minute_of_day', 'ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd', 't2_p40_h30', 't2_p40_h60', 't2_p50_h30']
Index: DatetimeIndex | TZ: America/New

In [50]:
import json
import os
from pathlib import Path

# Cargar JSON
with open(IN_SUMMARY, "r") as f:
    summary = json.load(f)

# Extraer columnas
features_col = summary["column_groups"]["features"]
targets_col  = summary["column_groups"]["targets"]
aux_col      = summary["column_groups"]["auxiliary"]

# Check rápido
print("Features:", features_col)
print("Targets:", targets_col)
print("Aux:", aux_col)

Features: ['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd']
Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
Aux: ['date', 'minute_of_day', 'regime_id']


In [51]:
# ==========================================================
# Diccionario con todos los datasets cargados
# ==========================================================
datasets = {
    "mnq_t2_train": mnq_t2_train,
    "mnq_t2_valid": mnq_t2_valid,
    "mnq_t2_test":  mnq_t2_test,
}

## **1.4. Verificación de orden temporal**

In [52]:
def comprobar_orden_datasets_base(**datasets):
    print("=" * 80)
    print("VALIDACIÓN ORDEN TEMPORAL - DATASETS BASE")
    print("=" * 80)

    for name, df in datasets.items():
        ordenado = df[["date", "minute_of_day"]].reset_index(drop=True).equals(
            df.sort_values(["date", "minute_of_day"])[["date", "minute_of_day"]].reset_index(drop=True)
        )

        print(f"{name}: {'OK' if ordenado else 'ERROR'}")

    print("=" * 80)

In [53]:
comprobar_orden_datasets_base(
    mnq_t2_train=mnq_t2_train,
    mnq_t2_valid=mnq_t2_valid,
    mnq_t2_test=mnq_t2_test,
)

VALIDACIÓN ORDEN TEMPORAL - DATASETS BASE
mnq_t2_train: OK
mnq_t2_valid: OK
mnq_t2_test: OK


# **2. Determinación del tamaño de ventana (window_size)**

El tamaño de ventana define la cantidad de historia que el modelo utiliza como entrada para cada predicción. En un esquema secuencial, cada muestra se construye como:

* X: secuencia de longitud L (window_size)
* y: target en el instante final de la ventana

En este proyecto, el tamaño de ventana no depende del target, sino de las features utilizadas.

---

**Relación entre window_size y features**

Las features incorporan memoria histórica a través de indicadores técnicos. En particular, la feature con mayor lookback es:

* roc_60 → requiere 60 minutos de historia

Esto implica que el dataset contiene información agregada de al menos 60 minutos en algunas variables. Por lo tanto:

* window_size < 60 → el modelo ve solo una parte de la dinámica temporal explícita
* window_size ≥ 60 → el modelo puede capturar la evolución completa consistente con las features

---

**Uso de ventanas menores al lookback**

Utilizar una ventana menor (por ejemplo, 30) no es incorrecto, ya que las features ya contienen información histórica agregada. Sin embargo:

* se reduce la capacidad del modelo para aprender patrones temporales completos
* se depende más de la “memoria implícita” de las features
* se pierde contexto temporal explícito

Este efecto es más relevante en modelos secuenciales (GRU, LSTM) que en modelos tabulares (XGBoost, Logistic), donde cada fila ya resume la información.

---

**Memoria implícita vs memoria explícita**

Existen dos fuentes de memoria en el modelo:

* memoria implícita: incorporada en las features (roc, ema, etc.)
* memoria explícita: longitud de la ventana (window_size)

Un window_size pequeño desplaza el peso hacia la memoria implícita; uno mayor permite al modelo aprender directamente la dinámica temporal.

---

**Estrategia adoptada**

Se evaluarán múltiples tamaños de ventana para capturar distintos niveles de contexto temporal:

* window_size = 30
* window_size = 60
* window_size = 90

Para cada tamaño de ventana se entrenarán modelos independientes sobre los mismos targets:

* t2_p40_h30
* t2_p40_h60
* t2_p50_h30

---

**Objetivo**

El objetivo de esta estrategia es determinar empíricamente:

* si ventanas más cortas (30) son suficientes
* si ventanas alineadas con el mayor lookback (60) mejoran el desempeño
* si ventanas más largas (90) capturan dinámica adicional útil

**Conclusión**

- El tamaño de ventana no se fija de forma arbitraria, sino que se valida experimentalmente.
- Se utilizarán tres configuraciones (30, 60, 90) y se comparará su desempeño para seleccionar la mejor combinación entre contexto temporal y capacidad predictiva.


## **2.1. Implementación**

In [54]:
import pandas as pd
from typing import Optional, List, Dict, Any


# ============================================================
# Validaciones base para datasets intradía
# ============================================================

def validate_intraday_dataset_for_windows(
    df: pd.DataFrame,
    *,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    required_cols: Optional[List[str]] = None,
    check_nans_in: Optional[List[str]] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Valida que un dataset intradía esté en condiciones de usarse
    para construir ventanas temporales.

    Qué verifica
    -------------
    1. Que existan las columnas requeridas.
    2. Que el dataset esté ordenado por (date, minute_of_day).
    3. Que no existan duplicados por (date, minute_of_day).
    4. Opcionalmente, que no existan NaNs en ciertas columnas clave.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset a validar.
    date_col : str
        Columna que identifica la sesión/día.
    minute_col : str
        Columna de minuto intradía.
    required_cols : list[str] | None
        Columnas que deben existir obligatoriamente.
    check_nans_in : list[str] | None
        Columnas donde se desea verificar ausencia de NaNs.
    verbose : bool
        Si True, imprime el detalle de las verificaciones.

    Retorna
    -------
    dict
        Resumen de validación.
    """

    def _print(msg: str) -> None:
        if verbose:
            print(msg)

    _print("=" * 80)
    _print("VALIDACIÓN BASE DEL DATASET")
    _print("=" * 80)

    # --------------------------------------------------------
    # 1) Verificar columnas requeridas
    # --------------------------------------------------------
    required = [date_col, minute_col]
    if required_cols is not None:
        required += list(required_cols)

    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

    _print(f"[OK] Columnas requeridas presentes: {required}")

    # --------------------------------------------------------
    # 2) Verificar orden por (date, minute_of_day)
    # --------------------------------------------------------
    check = df[[date_col, minute_col]].copy()
    check[date_col] = pd.to_datetime(check[date_col])

    check_sorted = check.sort_values([date_col, minute_col], kind="mergesort")

    is_sorted = check.reset_index(drop=True).equals(
        check_sorted.reset_index(drop=True)
    )

    if not is_sorted:
        raise ValueError(
            f"El dataset NO está ordenado por ({date_col}, {minute_col})"
        )

    _print(f"[OK] Orden correcto por ({date_col}, {minute_col})")

    # --------------------------------------------------------
    # 3) Verificar duplicados por timestamp intradía
    # --------------------------------------------------------
    n_duplicates = int(check.duplicated(subset=[date_col, minute_col]).sum())

    if n_duplicates > 0:
        raise ValueError(
            f"Se encontraron {n_duplicates} duplicados por ({date_col}, {minute_col})"
        )

    _print(f"[OK] Sin duplicados por ({date_col}, {minute_col})")

    # --------------------------------------------------------
    # 4) Verificar NaNs en columnas clave, si se solicita
    # --------------------------------------------------------
    nan_report = {}
    if check_nans_in is not None:
        for col in check_nans_in:
            if col not in df.columns:
                raise ValueError(f"La columna '{col}' no existe para chequeo de NaNs")

            n_nan = int(df[col].isna().sum())
            nan_report[col] = n_nan

            if n_nan > 0:
                raise ValueError(f"La columna '{col}' tiene {n_nan} NaNs")

        _print(f"[OK] Sin NaNs en columnas clave: {check_nans_in}")

    # --------------------------------------------------------
    # 5) Resumen
    # --------------------------------------------------------
    n_days = int(pd.to_datetime(df[date_col]).dt.date.nunique())

    _print(f"[INFO] Filas         : {len(df):,}")
    _print(f"[INFO] Días únicos   : {n_days:,}")
    _print("=" * 80)

    return {
        "n_rows": int(len(df)),
        "n_days": n_days,
        "is_sorted": True,
        "n_duplicates_timestamp": n_duplicates,
        "nan_report": nan_report,
    }


# ============================================================
# Regla de compatibilidad para ventanas
# ============================================================

def check_window_viability(
    df: pd.DataFrame,
    *,
    window_size: int,
    horizon: int,
    mode: str = "seq2one",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    required_cols: Optional[List[str]] = None,
    check_nans_in: Optional[List[str]] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Verifica si un tamaño de ventana es viable para un dataset dado.

    Lógica
    ------
    - seq2one:
        requiere al menos L observaciones por día
    - seq2seq:
        requiere al menos L + H observaciones por día

    Nota importante
    ---------------
    Para targets T2 ya precomputados en el dataset, en seq2one la
    restricción estructural correcta sigue siendo solo L.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset ya filtrado por split.
    window_size : int
        Lookback L.
    horizon : int
        Horizonte H del target.
    mode : str
        "seq2one" o "seq2seq".
    date_col : str
        Columna de sesión.
    minute_col : str
        Columna de minuto intradía.
    required_cols : list[str] | None
        Columnas extra requeridas para validar.
    check_nans_in : list[str] | None
        Columnas en las que se quiere exigir ausencia de NaNs.
    verbose : bool
        Si True, imprime verificaciones y resumen.

    Retorna
    -------
    dict
        Resumen de viabilidad de la ventana.
    """

    def _print(msg: str) -> None:
        if verbose:
            print(msg)

    if mode not in ["seq2one", "seq2seq"]:
        raise ValueError("mode debe ser 'seq2one' o 'seq2seq'")

    # --------------------------------------------------------
    # 1) Validación base del dataset antes del cálculo
    # --------------------------------------------------------
    base_validation = validate_intraday_dataset_for_windows(
        df,
        date_col=date_col,
        minute_col=minute_col,
        required_cols=required_cols,
        check_nans_in=check_nans_in,
        verbose=verbose,
    )

    # --------------------------------------------------------
    # 2) Longitud mínima requerida por día
    # --------------------------------------------------------
    required_length = window_size if mode == "seq2one" else window_size + horizon

    # Cantidad de filas por sesión
    counts_per_day = df.groupby(date_col).size().sort_index()

    # Días válidos e inválidos
    valid_days = counts_per_day[counts_per_day >= required_length]
    invalid_days = counts_per_day[counts_per_day < required_length]

    # Ventanas por día:
    # si un día tiene N observaciones y se requiere R,
    # entonces aporta N - R + 1 ventanas
    n_windows_per_day = (valid_days - required_length + 1).clip(lower=0)

    n_total_windows = int(n_windows_per_day.sum())

    # Algunas métricas útiles adicionales
    pct_valid_days = float(len(valid_days) / len(counts_per_day)) if len(counts_per_day) > 0 else 0.0
    avg_windows_per_valid_day = float(n_windows_per_day.mean()) if len(n_windows_per_day) > 0 else 0.0
    min_obs_day = int(counts_per_day.min()) if len(counts_per_day) > 0 else 0
    max_obs_day = int(counts_per_day.max()) if len(counts_per_day) > 0 else 0

    # --------------------------------------------------------
    # 3) Impresión de resultados
    # --------------------------------------------------------
    _print("=" * 80)
    _print("RESUMEN DE VIABILIDAD DE VENTANA")
    _print("=" * 80)
    _print(f"Modo                     : {mode}")
    _print(f"Window size (L)          : {window_size}")
    _print(f"Horizon (H)              : {horizon}")
    _print(f"Required length por día  : {required_length}")
    _print("-" * 80)
    _print(f"Días totales             : {len(counts_per_day)}")
    _print(f"Días válidos             : {len(valid_days)}")
    _print(f"Días descartados         : {len(invalid_days)}")
    _print(f"% días válidos           : {pct_valid_days:.2%}")
    _print("-" * 80)
    _print(f"Obs mín por día          : {min_obs_day}")
    _print(f"Obs máx por día          : {max_obs_day}")
    _print(f"Ventanas totales         : {n_total_windows}")
    _print(f"Prom ventanas/día válido : {avg_windows_per_valid_day:.2f}")
    _print("=" * 80)

    return {
        "window_size": window_size,
        "horizon": horizon,
        "mode": mode,
        "required_length": required_length,
        "n_rows": base_validation["n_rows"],
        "n_days_total": int(len(counts_per_day)),
        "n_days_valid": int(len(valid_days)),
        "n_days_invalid": int(len(invalid_days)),
        "pct_days_valid": pct_valid_days,
        "min_obs_day": min_obs_day,
        "max_obs_day": max_obs_day,
        "n_total_windows": n_total_windows,
        "avg_windows_per_valid_day": avg_windows_per_valid_day,
    }


# ============================================================
# Evaluación masiva para múltiples datasets y múltiples L
# ============================================================

def evaluate_window_grid(
    datasets: Dict[str, tuple],
    window_sizes: List[int],
    *,
    mode: str = "seq2one",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    required_cols: Optional[List[str]] = None,
    check_nans_in: Optional[List[str]] = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Evalúa una grilla de tamaños de ventana sobre múltiples datasets.

    Parámetros
    ----------
    datasets : dict
        Formato:
        {
            "nombre_dataset": (df, horizon),
            ...
        }
    window_sizes : list[int]
        Lista de ventanas L a evaluar.
    mode : str
        "seq2one" o "seq2seq".
    date_col : str
        Columna de sesión.
    minute_col : str
        Columna de minuto intradía.
    required_cols : list[str] | None
        Columnas extra requeridas.
    check_nans_in : list[str] | None
        Columnas a chequear sin NaNs.
    verbose : bool
        Si True, imprime detalle de cada corrida.

    Retorna
    -------
    pd.DataFrame
        Tabla consolidada con todos los resultados.
    """

    rows = []

    for dataset_name, (df, horizon) in datasets.items():

        if verbose:
            print("\n" + "=" * 90)
            print(f"DATASET: {dataset_name} | H={horizon}")
            print("=" * 90)

        for L in window_sizes:
            result = check_window_viability(
                df,
                window_size=L,
                horizon=horizon,
                mode=mode,
                date_col=date_col,
                minute_col=minute_col,
                required_cols=required_cols,
                check_nans_in=check_nans_in,
                verbose=verbose,
            )
            result["dataset"] = dataset_name
            rows.append(result)

    summary_df = pd.DataFrame(rows)

    # Reordenar columnas para lectura más clara
    preferred_cols = [
        "dataset",
        "mode",
        "window_size",
        "horizon",
        "required_length",
        "n_rows",
        "n_days_total",
        "n_days_valid",
        "n_days_invalid",
        "pct_days_valid",
        "min_obs_day",
        "max_obs_day",
        "n_total_windows",
        "avg_windows_per_valid_day",
    ]

    summary_df = summary_df[preferred_cols].sort_values(
        ["dataset", "window_size"]
    ).reset_index(drop=True)

    return summary_df

In [55]:
time_cols = ["date", "minute_of_day"]


In [56]:
targets_col

['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']

In [57]:
window_sizes = [30, 60, 90]

datasets = {
    "t2_p40_h30": (mnq_t2_train, 30),
    "t2_p40_h60": (mnq_t2_train, 60),
    "t2_p50_h30": (mnq_t2_train, 30),
}

required_cols = time_cols + features_col + targets_col

summary_windows_train = evaluate_window_grid(
    datasets=datasets,
    window_sizes=window_sizes,
    mode="seq2one",
    date_col="date",
    minute_col="minute_of_day",
    required_cols=required_cols,
    check_nans_in=features_col,
    verbose=True,
)

print(summary_windows_train)


DATASET: t2_p40_h30 | H=30
VALIDACIÓN BASE DEL DATASET
[OK] Columnas requeridas presentes: ['date', 'minute_of_day', 'date', 'minute_of_day', 'ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd', 't2_p40_h30', 't2_p40_h60', 't2_p50_h30']
[OK] Orden correcto por (date, minute_of_day)
[OK] Sin duplicados por (date, minute_of_day)
[OK] Sin NaNs en columnas clave: ['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd']
[INFO] Filas         : 62,220
[INFO] Días únicos   : 1,037
RESUMEN DE VIABILIDAD DE VENTANA
Modo                     : seq2one
Window size (L)          : 30
Horizon (H)              : 30
Required length por día  : 30
--------------------------------------------------------------------------------
Días totales             : 1037
Días válidos             : 1037
Días descartados         : 0
% días válidos           : 100.00%
--------------------------------------------------------------------------------
Obs mín por día          : 60


## **2.2. Observaciones sobre la viabilidad de ventanas**


**1. Estructura del dataset**

* Cada día contiene exactamente 60 observaciones
* Esto corresponde a una sesión intradía acotada (opening)
* La longitud temporal disponible por día es limitada

**2. Evaluación de tamaños de ventana**

Ventana L = 30

* 31 ventanas por día
* 100% de días válidos
* Dataset con buen volumen de muestras
  → Alta viabilidad

Ventana L = 60

* 1 ventana por día
* 100% de días válidos
* Dataset extremadamente reducido
  → Viabilidad técnica, pero poco útil para entrenamiento

Ventana L = 90

* 0 días válidos
* 0 ventanas generadas
  → No viable

**3. Implicación estructural**
   
   El filtrado al régimen 2 mejora la calidad de la señal, pero reduce la longitud temporal disponible por sesión. Esto limita el tamaño máximo de ventana utilizable.

**4. Rol de las features**
   
   Las features incluyen indicadores con memoria larga (por ejemplo, roc_60, ema_60), que ya incorporan información de hasta 60 minutos.
   Esto implica que la memoria histórica no depende exclusivamente del tamaño de la ventana.

**5. Interpretación del tamaño de ventana**

* La ventana aporta dinámica temporal explícita
* Las features aportan memoria histórica implícita
  Una ventana más corta no implica pérdida total de contexto, ya que parte de la información ya está embebida en las features

**6. Criterios de selección**

Se elige L = 30 porque:

* Maximiza la cantidad de muestras disponibles
* Mantiene todos los días del dataset
* Permite capturar dinámica intradía reciente
* Evita datasets pequeños que dificultan el aprendizaje
* Es consistente con la memoria ya incorporada en las features

**7. Conclusión**

* L = 90 es inviable
* L = 60 es marginal y poco eficiente
* L = 30 es el único tamaño que ofrece un balance adecuado entre contexto temporal y tamaño de muestra

Decisión: avanzar con ventanas de tamaño 30 para la construcción de los modelos.



## **2.3. Generación de summary de ventanas**

In [58]:
import json
from pathlib import Path


def generate_window_analysis_summary(
    window_viability_df,
    window_sizes,
    out_path: Path,
    verbose: bool = True,
):
    """
    Genera un summary del análisis de viabilidad de ventanas.

    Contenido:
    - grilla de ventanas utilizada
    - verificación estructural del dataset
    - métricas agregadas por ventana
    - conclusiones clave para modelado
    """

    def _print(msg):
        if verbose:
            print(msg)

    # ---------------------------------------------------
    # 1) Resumen estructural global
    # ---------------------------------------------------
    structure_summary = {
        "constant_intraday_length": bool(
            (window_viability_df["min_obs_day"] == window_viability_df["max_obs_day"]).all()
        ),
        "min_obs_per_day": int(window_viability_df["min_obs_day"].min()),
        "max_obs_per_day": int(window_viability_df["max_obs_day"].max()),
        "n_days_total": int(window_viability_df["n_days_total"].max()),
    }

    # ---------------------------------------------------
    # 2) Métricas por tamaño de ventana
    # ---------------------------------------------------
    window_stats = {}
    viable_windows = []
    marginal_windows = []
    invalid_windows = []

    for L in window_sizes:
        subset = window_viability_df[window_viability_df["window_size"] == L]

        n_days_valid_mean = float(subset["n_days_valid"].mean())
        n_days_invalid_mean = float(subset["n_days_invalid"].mean())
        n_total_windows_mean = float(subset["n_total_windows"].mean())
        avg_windows_per_day = float(subset["avg_windows_per_valid_day"].mean())
        pct_days_valid = float(subset["pct_days_valid"].mean())

        is_structurally_viable = pct_days_valid == 1.0 and n_total_windows_mean > 0
        is_marginal = is_structurally_viable and avg_windows_per_day <= 1.0
        is_invalid = pct_days_valid == 0.0 or n_total_windows_mean == 0

        if is_invalid:
            invalid_windows.append(L)
        elif is_marginal:
            marginal_windows.append(L)
        else:
            viable_windows.append(L)

        window_stats[str(L)] = {
            "n_days_valid_mean": int(n_days_valid_mean),
            "n_days_invalid_mean": int(n_days_invalid_mean),
            "n_total_windows_mean": int(n_total_windows_mean),
            "avg_windows_per_day": avg_windows_per_day,
            "pct_days_valid": pct_days_valid,
            "is_structurally_viable": bool(is_structurally_viable),
            "is_marginal_for_training": bool(is_marginal),
            "is_invalid": bool(is_invalid),
        }

    # ---------------------------------------------------
    # 3) Verificaciones clave
    # ---------------------------------------------------
    checks = {
        "fixed_length_days": structure_summary["constant_intraday_length"],
        "at_least_one_viable_window": len(viable_windows) > 0,
        "contains_invalid_windows": len(invalid_windows) > 0,
        "contains_marginal_windows": len(marginal_windows) > 0,
        "consistent_window_generation": True,
    }

    # ---------------------------------------------------
    # 4) Conclusiones estructurales
    # ---------------------------------------------------
    conclusions = []

    if structure_summary["constant_intraday_length"]:
        conclusions.append(
            "El dataset presenta longitud intradía constante en todas las sesiones."
        )
    else:
        conclusions.append(
            "El dataset no presenta longitud intradía constante en todas las sesiones."
        )

    conclusions.append(
        f"Cada sesión contiene entre {structure_summary['min_obs_per_day']} y "
        f"{structure_summary['max_obs_per_day']} observaciones intradía."
    )

    if len(viable_windows) > 0:
        conclusions.append(
            f"Las ventanas estructuralmente viables para modelado son: {viable_windows}."
        )

    if len(marginal_windows) > 0:
        conclusions.append(
            f"Las ventanas {marginal_windows} son técnicamente válidas, "
            f"pero generan muy pocas muestras por día y resultan marginales para entrenamiento."
        )

    if len(invalid_windows) > 0:
        conclusions.append(
            f"Las ventanas {invalid_windows} no son viables con la longitud intradía actual del dataset."
        )

    conclusions.append(
        "En modo seq2one, el horizonte del target no afecta la viabilidad estructural de la ventana."
    )

    if 30 in viable_windows:
        conclusions.append(
            "La ventana de 30 minutos ofrece el mejor balance entre contexto temporal y cantidad de muestras."
        )

    if 60 in marginal_windows:
        conclusions.append(
            "La ventana de 60 minutos queda al límite de la longitud disponible por sesión y produce una sola ventana por día."
        )

    if 90 in invalid_windows:
        conclusions.append(
            "La ventana de 90 minutos excede la longitud intradía disponible y no puede utilizarse en este stage."
        )

    conclusions.append(
        "La selección final de la ventana debe considerar tanto viabilidad estructural como utilidad práctica para entrenamiento."
    )

    # ---------------------------------------------------
    # 5) Objeto final
    # ---------------------------------------------------
    summary = {
        "window_grid": window_sizes,
        "structure_summary": structure_summary,
        "window_statistics": window_stats,
        "checks": checks,
        "viable_windows": viable_windows,
        "marginal_windows": marginal_windows,
        "invalid_windows": invalid_windows,
        "recommended_window": 30 if 30 in viable_windows else None,
        "conclusions": conclusions,
    }

    # ---------------------------------------------------
    # 6) Guardado
    # ---------------------------------------------------
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=4, ensure_ascii=False)

    _print("=" * 100)
    _print("WINDOW ANALYSIS SUMMARY GENERADO")
    _print(f"Path: {out_path}")
    _print("=" * 100)

    return summary

In [59]:
OUT_WINDOW_SUMMARY = DRIVE_DIR / "data/07_windows/window_analysis_summary.json"

window_summary = generate_window_analysis_summary(
    window_viability_df=summary_windows_train,
    window_sizes=[30, 60, 90],
    out_path=OUT_WINDOW_SUMMARY,
    verbose=True,
)

WINDOW ANALYSIS SUMMARY GENERADO
Path: /content/drive/MyDrive/neural_profit/data/07_windows/window_analysis_summary.json


In [60]:
window_summary

{'window_grid': [30, 60, 90],
 'structure_summary': {'constant_intraday_length': True,
  'min_obs_per_day': 60,
  'max_obs_per_day': 60,
  'n_days_total': 1037},
 'window_statistics': {'30': {'n_days_valid_mean': 1037,
   'n_days_invalid_mean': 0,
   'n_total_windows_mean': 32147,
   'avg_windows_per_day': 31.0,
   'pct_days_valid': 1.0,
   'is_structurally_viable': True,
   'is_marginal_for_training': False,
   'is_invalid': False},
  '60': {'n_days_valid_mean': 1037,
   'n_days_invalid_mean': 0,
   'n_total_windows_mean': 1037,
   'avg_windows_per_day': 1.0,
   'pct_days_valid': 1.0,
   'is_structurally_viable': True,
   'is_marginal_for_training': True,
   'is_invalid': False},
  '90': {'n_days_valid_mean': 0,
   'n_days_invalid_mean': 1037,
   'n_total_windows_mean': 0,
   'avg_windows_per_day': 0.0,
   'pct_days_valid': 0.0,
   'is_structurally_viable': False,
   'is_marginal_for_training': False,
   'is_invalid': True}},
 'checks': {'fixed_length_days': True,
  'at_least_one_viab

# **3. Escalado de los datasets**

## **3.1. Introducción teórica y principios de escalado**


El escalado constituye una etapa crítica dentro del pipeline de modelado, ya que la mayoría de los algoritmos de *machine learning* son sensibles a la escala de las variables de entrada. Diferencias en magnitud pueden provocar que ciertas features dominen el proceso de aprendizaje, afectando negativamente tanto la convergencia como el desempeño del modelo.

El dataset presenta heterogeneidad en las variables de entrada. Conviven indicadores técnicos, variables temporales y variables derivadas, cada una con rangos y distribuciones distintas. Esta diversidad requiere un tratamiento uniforme que evite sesgos inducidos por escala y permita una comparación adecuada entre variables.

El proceso de escalado debe respetar estrictamente la coherencia temporal del problema. En particular, el scaler se ajusta exclusivamente utilizando el conjunto de entrenamiento y luego se aplica sin modificación a los conjuntos de validación y prueba. Este procedimiento evita la introducción de información futura y garantiza la validez de la evaluación.

El escalado se aplica únicamente sobre variables continuas, incluyendo indicadores técnicos, variables derivadas y la variable temporal `minute_of_day`. No se transforman variables categóricas o discretas, como `regime_id`, ni columnas de identificación o fecha, ya que su significado no depende de la magnitud sino de su valor categórico o estructural.

El conjunto de features a escalar se define explícitamente y se mantiene consistente en todo el pipeline. Esto asegura reproducibilidad, trazabilidad y comparabilidad entre experimentos, especialmente al evaluar distintos modelos o configuraciones de ventana.

El escalado se realiza sobre los datos en formato tabular, antes de la generación de ventanas. Este orden permite conservar los nombres de las columnas, seleccionar de forma explícita las variables a transformar y evitar errores una vez que los datos son vectorizados.

El método de escalado debe ser adecuado para la naturaleza de los datos. En este caso, el uso de `StandardScaler` resulta apropiado, ya que centra las variables en media cero y varianza unitaria, facilitando el entrenamiento de modelos lineales, redes neuronales y algoritmos basados en distancia.

El escalado se integra como una transformación determinista dentro del pipeline. Esto implica guardar el scaler ajustado y reutilizarlo en todas las etapas posteriores, asegurando consistencia entre entrenamiento, validación, prueba e inferencia.

El objetivo del escalado no es alterar la información contenida en las features, sino proyectarlas a un espacio numérico comparable. Esto permite que el modelo aprenda relaciones estables entre variables, evita que ciertas features sean priorizadas artificialmente por su magnitud y mejora la capacidad de generalización entre distintos días y regímenes intradía.

Con estos criterios establecidos, el siguiente paso consiste en implementar el escalado de forma explícita y reproducible para los conjuntos `mnq_train`, `mnq_valid` y `mnq_test`.


## **3.2. Implementación de escalado**

### **3.2.1. Función para elegir escalador**


In [61]:
from __future__ import annotations

import json
import joblib
import pandas as pd
from pathlib import Path
from typing import Optional, Tuple, Dict, Any, Sequence
from sklearn.base import TransformerMixin
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler


# ============================================================
# 1. Selección de scaler
# ============================================================
def choose_scaler(
    scaler_type: str = "standard",
    *,
    # StandardScaler
    with_mean: bool = True,
    with_std: bool = True,
    # MinMaxScaler
    feature_range: tuple[float, float] = (0.0, 1.0),
    # RobustScaler
    quantile_range: tuple[float, float] = (25.0, 75.0),
    with_centering: bool = True,
    with_scaling: bool = True,
) -> Optional[TransformerMixin]:
    """
    Devuelve un scaler de sklearn según `scaler_type`.
    """
    st = (scaler_type or "").strip().lower()

    if st in {"standard", "z", "zscore"}:
        return StandardScaler(with_mean=with_mean, with_std=with_std)

    if st in {"minmax", "min_max"}:
        return MinMaxScaler(feature_range=feature_range)

    if st == "robust":
        return RobustScaler(
            quantile_range=quantile_range,
            with_centering=with_centering,
            with_scaling=with_scaling,
        )

    if st in {"none", "passthrough"}:
        return None

    raise ValueError(
        f"scaler_type inválido: '{scaler_type}'. "
        "Use: 'standard', 'minmax', 'robust' o 'none'."
    )

### **3.2.2. Función para escalar datasets train, valid y test**


In [62]:
import pandas as pd
from typing import Tuple, Dict, Any, Sequence


# ============================================================
# 2. Validación de orden temporal
# ============================================================
def validar_orden_temporal(
    df: pd.DataFrame,
    *,
    name: str = "dataset",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
) -> None:
    """
    Valida que el DataFrame esté ordenado temporalmente por:
    1) date ascendente
    2) minute_of_day ascendente dentro de cada date
    """
    if date_col not in df.columns:
        raise KeyError(f"{name}: no existe la columna '{date_col}'")

    if minute_col not in df.columns:
        raise KeyError(f"{name}: no existe la columna '{minute_col}'")

    ordenado = (
        df[[date_col, minute_col]]
        .reset_index(drop=True)
        .equals(
            df[[date_col, minute_col]]
            .sort_values([date_col, minute_col], kind="mergesort")
            .reset_index(drop=True)
        )
    )

    if not ordenado:
        raise ValueError(
            f"{name}: el dataset NO está ordenado por [{date_col}, {minute_col}]"
        )


In [63]:
# ============================================================
# 3. Escalado sin leakage
# ============================================================
def scale_mnq_splits(
    mnq_train: pd.DataFrame,
    mnq_valid: pd.DataFrame,
    mnq_test: pd.DataFrame,
    *,
    scaler,
    features_to_scale: Sequence[str],
    date_col: str = "date",
    minute_col: str = "minute_of_day",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """
    Escala los splits evitando data leakage.

    Reglas
    ------
    - Valida orden temporal en train/valid/test.
    - Exige mismas columnas y mismo orden en los tres splits.
    - Fit del scaler SOLO en train.
    - Transform sobre train/valid/test usando las mismas columnas.
    - Escala únicamente las columnas de `features_to_scale`.
    """

    # ------------------------------------------------------------
    # 0. Validación de orden temporal
    # ------------------------------------------------------------
    validar_orden_temporal(
        mnq_train, name="mnq_train", date_col=date_col, minute_col=minute_col
    )
    validar_orden_temporal(
        mnq_valid, name="mnq_valid", date_col=date_col, minute_col=minute_col
    )
    validar_orden_temporal(
        mnq_test, name="mnq_test", date_col=date_col, minute_col=minute_col
    )

    # ------------------------------------------------------------
    # 1. Validación de columnas idénticas
    # ------------------------------------------------------------
    cols_train = list(mnq_train.columns)
    if list(mnq_valid.columns) != cols_train or list(mnq_test.columns) != cols_train:
        raise ValueError(
            "mnq_train, mnq_valid y mnq_test deben tener exactamente las mismas columnas y orden."
        )

    # ------------------------------------------------------------
    # 2. Validar que todas las columnas a escalar existan
    # ------------------------------------------------------------
    missing_scale_cols = [c for c in features_to_scale if c not in cols_train]
    if missing_scale_cols:
        raise ValueError(
            f"Faltan columnas esperadas en features_to_scale: {missing_scale_cols}"
        )

    scale_cols = list(features_to_scale)

    # ------------------------------------------------------------
    # 3. Validar NaNs en columnas a escalar
    # ------------------------------------------------------------
    for split_name, df_ in {
        "mnq_train": mnq_train,
        "mnq_valid": mnq_valid,
        "mnq_test": mnq_test,
    }.items():
        nan_counts = df_[scale_cols].isna().sum()
        nan_cols = nan_counts[nan_counts > 0]
        if len(nan_cols) > 0:
            raise ValueError(
                f"{split_name}: hay NaNs en columnas a escalar: {nan_cols.to_dict()}"
            )

    # ------------------------------------------------------------
    # 4. Si no hay scaler, devolver copias sin transformación
    # ------------------------------------------------------------
    if scaler is None:
        meta = {
            "scaled": False,
            "reason": "Scaler=None",
            "scale_cols": scale_cols,
            "temporal_order_validated": True,
        }
        return mnq_train.copy(), mnq_valid.copy(), mnq_test.copy(), meta

    tr = mnq_train.copy()
    va = mnq_valid.copy()
    te = mnq_test.copy()

    # ------------------------------------------------------------
    # 5. Fit SOLO en train
    # ------------------------------------------------------------
    scaler.fit(tr[scale_cols])

    # ------------------------------------------------------------
    # 6. Transform en train / valid / test
    # ------------------------------------------------------------
    tr.loc[:, scale_cols] = scaler.transform(tr[scale_cols])
    va.loc[:, scale_cols] = scaler.transform(va[scale_cols])
    te.loc[:, scale_cols] = scaler.transform(te[scale_cols])

    # ------------------------------------------------------------
    # 7. Metadata del escalado
    # ------------------------------------------------------------
    meta = {
        "scaled": True,
        "scaler_class": scaler.__class__.__name__,
        "scale_cols": scale_cols,
        "temporal_order_validated": True,
    }

    return tr, va, te, meta

### **3.2.3. Guardar dataset escalados**


In [64]:
import json
import joblib
import pandas as pd
from pathlib import Path
from typing import Tuple, Dict, Any, Sequence


def save_scaled_datasets(
    *,
    mnq_train_scaled: pd.DataFrame,
    mnq_valid_scaled: pd.DataFrame,
    mnq_test_scaled: pd.DataFrame,
    scale_meta: Dict[str, Any],
    out_train_path: Path,
    out_valid_path: Path,
    out_test_path: Path,
    out_meta_path: Path,
    scaler=None,
    out_scaler_path: Path | None = None,
) -> None:
    """
    Guarda datasets escalados, metadata y opcionalmente el scaler.

    Notas
    -----
    - Conserva el índice del DataFrame.
    - Compatible con arquitectura basada en rutas externas.
    """

    # ------------------------------------------------------------
    # 1) Crear directorios
    # ------------------------------------------------------------
    for path in [out_train_path, out_valid_path, out_test_path, out_meta_path]:
        path.parent.mkdir(parents=True, exist_ok=True)

    if scaler is not None and out_scaler_path is not None:
        out_scaler_path.parent.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------
    # 2) Guardar datasets escalados preservando índice
    # ------------------------------------------------------------
    mnq_train_scaled.to_parquet(out_train_path, index=True)
    mnq_valid_scaled.to_parquet(out_valid_path, index=True)
    mnq_test_scaled.to_parquet(out_test_path, index=True)

    # ------------------------------------------------------------
    # 3) Preparar metadata sin mutar el dict original
    # ------------------------------------------------------------
    meta_to_save = dict(scale_meta)
    meta_to_save["temporal_order_validated"] = True
    meta_to_save["sorted_by"] = ["date", "minute_of_day"]

    # ------------------------------------------------------------
    # 4) Guardar metadata
    # ------------------------------------------------------------
    with out_meta_path.open("w", encoding="utf-8") as f:
        json.dump(meta_to_save, f, indent=2, ensure_ascii=False)

    # ------------------------------------------------------------
    # 5) Guardar scaler si corresponde
    # ------------------------------------------------------------
    if scaler is not None and out_scaler_path is not None:
        joblib.dump(scaler, out_scaler_path)


def load_or_scale_mnq_datasets(
    *,
    mnq_train: pd.DataFrame,
    mnq_valid: pd.DataFrame,
    mnq_test: pd.DataFrame,
    scaler,
    features_to_scale: Sequence[str],
    out_train_path: Path,
    out_valid_path: Path,
    out_test_path: Path,
    out_meta_path: Path,
    out_scaler_path: Path,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any], Any]:
    """
    Carga datasets escalados y scaler si ya existen.
    Si no existen, realiza el escalado (fit SOLO en train), guarda artefactos
    y retorna datasets escalados + metadata + scaler.

    Retorna
    -------
    mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler
    """

    paths = {
        "train": out_train_path,
        "valid": out_valid_path,
        "test": out_test_path,
        "meta": out_meta_path,
        "scaler": out_scaler_path,
    }

    if verbose:
        print("Verificando existencia de datasets escalados y scaler...")
        for k, p in paths.items():
            print(f"  - {k}: {'OK' if p.exists() else 'NO EXISTE'}")

    files_exist = all(p.exists() for p in paths.values())

    # ------------------------------------------------------------
    # Caso 1: todo existe -> cargar
    # ------------------------------------------------------------
    if files_exist:
        if verbose:
            print("\nTodos los archivos existen. Cargando artefactos desde disco...")

        mnq_train_s = pd.read_parquet(out_train_path)
        mnq_valid_s = pd.read_parquet(out_valid_path)
        mnq_test_s = pd.read_parquet(out_test_path)

        with out_meta_path.open("r", encoding="utf-8") as f:
            scale_meta = json.load(f)

        scaler_loaded = joblib.load(out_scaler_path)

        if verbose:
            print("Carga completada. No se recalculó el escalado.")

        return mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler_loaded

    # ------------------------------------------------------------
    # Caso 2: falta algún archivo -> recalcular
    # ------------------------------------------------------------
    if verbose:
        print("\nNo se encontraron todos los artefactos necesarios.")
        print("Recalculando escalado desde cero (fit SOLO en train)...")

    mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta = scale_mnq_splits(
        mnq_train,
        mnq_valid,
        mnq_test,
        scaler=scaler,
        features_to_scale=features_to_scale,
        date_col=date_col,
        minute_col=minute_col,
    )

    save_scaled_datasets(
        mnq_train_scaled=mnq_train_s,
        mnq_valid_scaled=mnq_valid_s,
        mnq_test_scaled=mnq_test_s,
        scale_meta=scale_meta,
        out_train_path=out_train_path,
        out_valid_path=out_valid_path,
        out_test_path=out_test_path,
        out_meta_path=out_meta_path,
        scaler=scaler,
        out_scaler_path=out_scaler_path,
    )

    if verbose:
        print("Escalado completo y persistido en disco.")

    return mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler


## **3.3. Aplicación**


In [65]:
from pathlib import Path

# ============================================================
# 1. Features a escalar
# ============================================================
features_to_scale = [
    #"minute_of_day",
    "ema_60",
    "roc_60",
    "roc_30",
    "stoch_k_30",
    "mom_5",
    "atr_norm_10",
    "macd",
]

print("Features a escalar:")
print(features_to_scale)


# ============================================================
# 2. Elegir scaler
# ============================================================
scaler = choose_scaler(
    scaler_type="standard"
)


# ============================================================
# 3. Definir paths
# ============================================================
OUT_DIR_SCALED = DRIVE_DIR / "data/06_scaled"

OUT_TRAIN_PATH = OUT_DIR_SCALED / "mnq_t2_train_scaled.parquet"
OUT_VALID_PATH = OUT_DIR_SCALED / "mnq_t2_valid_scaled.parquet"
OUT_TEST_PATH  = OUT_DIR_SCALED / "mnq_t2_test_scaled.parquet"
OUT_META_PATH  = OUT_DIR_SCALED / "scaling_meta_mnq_t2.json"
OUT_SCALER_PATH = OUT_DIR_SCALED / "scaler_mnq_t2.pkl"


# ============================================================
# 4. Cargar o escalar datasets
# ============================================================
mnq_t2_train_scaled, mnq_t2_valid_scaled, mnq_t2_test_scaled, scale_meta, scaler_fitted = load_or_scale_mnq_datasets(
    mnq_train=mnq_t2_train,
    mnq_valid=mnq_t2_valid,
    mnq_test=mnq_t2_test,
    scaler=scaler,
    features_to_scale=features_to_scale,
    out_train_path=OUT_TRAIN_PATH,
    out_valid_path=OUT_VALID_PATH,
    out_test_path=OUT_TEST_PATH,
    out_meta_path=OUT_META_PATH,
    out_scaler_path=OUT_SCALER_PATH,
    date_col="date",
    minute_col="minute_of_day",
    verbose=True,
)


# ============================================================
# 5. Resumen final
# ============================================================
print("=" * 100)
print("ESCALADO FINALIZADO")
print("=" * 100)
print(f"Train escalado : {OUT_TRAIN_PATH}")
print(f"Valid escalado : {OUT_VALID_PATH}")
print(f"Test escalado  : {OUT_TEST_PATH}")
print(f"Meta           : {OUT_META_PATH}")
print(f"Scaler         : {OUT_SCALER_PATH}")
print("=" * 100)
print("Metadata:")
print(scale_meta)

Features a escalar:
['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd']
Verificando existencia de datasets escalados y scaler...
  - train: OK
  - valid: OK
  - test: OK
  - meta: OK
  - scaler: OK

Todos los archivos existen. Cargando artefactos desde disco...
Carga completada. No se recalculó el escalado.
ESCALADO FINALIZADO
Train escalado : /content/drive/MyDrive/neural_profit/data/06_scaled/mnq_t2_train_scaled.parquet
Valid escalado : /content/drive/MyDrive/neural_profit/data/06_scaled/mnq_t2_valid_scaled.parquet
Test escalado  : /content/drive/MyDrive/neural_profit/data/06_scaled/mnq_t2_test_scaled.parquet
Meta           : /content/drive/MyDrive/neural_profit/data/06_scaled/scaling_meta_mnq_t2.json
Scaler         : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl
Metadata:
{'scaled': True, 'scaler_class': 'StandardScaler', 'scale_cols': ['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd'], 'temporal_order_valida

## **3.4. Verificación de orden temporal**


In [66]:
def verificar_orden_dataset(
    target_name: str,
    df: pd.DataFrame,
    meta: dict,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
):
    """
    Verifica que el dataset esté ordenado temporalmente y que la metadata
    registre la validación temporal.
    """

    print(f"\n=== Verificación temporal: {target_name} ===")

    # -------------------------------------------------------
    # 1. Verificar metadata
    # -------------------------------------------------------
    meta_flag = meta.get("temporal_order_validated", False)

    print("Metadata temporal_order_validated:", meta_flag)

    # -------------------------------------------------------
    # 2. Verificar orden real del dataframe
    # -------------------------------------------------------
    ordered = (
        df[[date_col, minute_col]]
        .reset_index(drop=True)
        .equals(
            df[[date_col, minute_col]]
            .sort_values([date_col, minute_col])
            .reset_index(drop=True)
        )
    )

    print("Dataset ordenado temporalmente:", ordered)

    # -------------------------------------------------------
    # 3. Resultado final
    # -------------------------------------------------------
    if meta_flag and ordered:
        print("✔ Dataset temporalmente consistente")
    else:
        print("✘ Posible problema de orden temporal")

    return meta_flag, ordered

In [67]:
verificar_orden_dataset(
    "mnq_t2_train_scaled",
    mnq_t2_train_scaled,
    scale_meta,
)



=== Verificación temporal: mnq_t2_train_scaled ===
Metadata temporal_order_validated: True
Dataset ordenado temporalmente: True
✔ Dataset temporalmente consistente


(True, True)

## **3.5. Verificación de escalado**


In [68]:
import pandas as pd
import numpy as np
from typing import Dict, Any, Sequence


def verify_scaling(
    train_raw: pd.DataFrame,
    train_scaled: pd.DataFrame,
    valid_raw: pd.DataFrame,
    valid_scaled: pd.DataFrame,
    test_raw: pd.DataFrame,
    test_scaled: pd.DataFrame,
    *,
    name: str,
    target_col: str,
    features_to_scale: Sequence[str],
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    tol_mean: float = 0.05,
    tol_std: float = 0.05,
) -> Dict[str, Any]:
    """
    Verifica que el escalado se haya realizado correctamente.

    Chequeos:
    - TRAIN: media ~ 0 y std ~ 1 en columnas escaladas
    - VALID/TEST: fueron transformados
    - Target no fue modificado
    - Date no fue modificada
    - El orden de filas respecto al raw fue preservado
    - El dataset escalado sigue ordenado temporalmente
    - Índice (DatetimeIndex) preservado
    """

    print("\n" + "=" * 25 + f" {name} " + "=" * 25)

    # ------------------------------------------------------------
    # 1) Features presentes
    # ------------------------------------------------------------
    scale_cols = [c for c in features_to_scale if c in train_raw.columns]
    if not scale_cols:
        raise ValueError("No hay columnas válidas para verificar en features_to_scale")

    print(f"Features a verificar (presentes): {len(scale_cols)}")
    print(scale_cols)

    # ------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------
    def _same_col(a: pd.DataFrame, b: pd.DataFrame, col: str) -> bool:
        return np.array_equal(a[col].values, b[col].values)

    def _is_sorted(df: pd.DataFrame) -> bool:
        return df[[date_col, minute_col]].reset_index(drop=True).equals(
            df[[date_col, minute_col]]
            .sort_values([date_col, minute_col], kind="mergesort")
            .reset_index(drop=True)
        )

    def _same_row_order(a: pd.DataFrame, b: pd.DataFrame) -> bool:
        """
        Verifica que el escalado no haya alterado el orden de las filas.
        No compara minute_of_day exacto porque puede haber sido escalado.
        """
        same_len = len(a) == len(b)
        same_index = a.index.equals(b.index)
        same_date_sequence = a[date_col].reset_index(drop=True).equals(
            b[date_col].reset_index(drop=True)
        )
        return same_len and same_index and same_date_sequence

    def _check_index_equal(df_raw: pd.DataFrame, df_scaled: pd.DataFrame):
        if isinstance(df_raw.index, pd.DatetimeIndex) and isinstance(df_scaled.index, pd.DatetimeIndex):
            return df_raw.index.equals(df_scaled.index)
        return None

    def _transformation_report(raw: pd.DataFrame, scaled: pd.DataFrame, cols: Sequence[str]) -> Dict[str, Any]:
        changed_cols = []
        unchanged_cols = []

        for col in cols:
            equal = np.allclose(raw[col].values, scaled[col].values)
            if equal:
                unchanged_cols.append(col)
            else:
                changed_cols.append(col)

        return {
            "n_changed_cols": len(changed_cols),
            "n_unchanged_cols": len(unchanged_cols),
            "changed_cols": changed_cols,
            "unchanged_cols": unchanged_cols,
            "all_changed": len(unchanged_cols) == 0,
        }

    # ------------------------------------------------------------
    # 2) TRAIN: verificar estandarización
    # ------------------------------------------------------------
    means = train_scaled[scale_cols].mean()
    stds = train_scaled[scale_cols].std(ddof=0)

    mean_fail_cols = means.index[means.abs() > tol_mean].tolist()
    std_fail_cols = stds.index[(stds - 1).abs() > tol_std].tolist()

    print("\nTRAIN (estandarización):")
    print(f"  Mean fuera tolerancia (|mean| > {tol_mean}): {len(mean_fail_cols)}")
    print(f"  Std  fuera tolerancia (|std-1| > {tol_std}): {len(std_fail_cols)}")

    if mean_fail_cols:
        print(f"  Columnas con mean fuera de tolerancia: {mean_fail_cols}")
    if std_fail_cols:
        print(f"  Columnas con std fuera de tolerancia : {std_fail_cols}")

    # ------------------------------------------------------------
    # 3) VALID / TEST: verificar transformación
    # ------------------------------------------------------------
    valid_transform_report = _transformation_report(valid_raw, valid_scaled, scale_cols)
    test_transform_report = _transformation_report(test_raw, test_scaled, scale_cols)

    print("\nTransformación aplicada:")
    print(f"  VALID - columnas modificadas: {valid_transform_report['n_changed_cols']} / {len(scale_cols)}")
    print(f"  TEST  - columnas modificadas: {test_transform_report['n_changed_cols']} / {len(scale_cols)}")

    # ------------------------------------------------------------
    # 4) Target sin modificar
    # ------------------------------------------------------------
    target_same_train = _same_col(train_raw, train_scaled, target_col)
    target_same_valid = _same_col(valid_raw, valid_scaled, target_col)
    target_same_test = _same_col(test_raw, test_scaled, target_col)

    print("\nTarget sin modificar:")
    print(f"  TRAIN: {target_same_train}")
    print(f"  VALID: {target_same_valid}")
    print(f"  TEST : {target_same_test}")

    # ------------------------------------------------------------
    # 5) Date sin modificar
    # ------------------------------------------------------------
    date_same_train = _same_col(train_raw, train_scaled, date_col)
    date_same_valid = _same_col(valid_raw, valid_scaled, date_col)
    date_same_test = _same_col(test_raw, test_scaled, date_col)

    print("\nDate sin modificar:")
    print(f"  TRAIN: {date_same_train}")
    print(f"  VALID: {date_same_valid}")
    print(f"  TEST : {date_same_test}")

    # ------------------------------------------------------------
    # 6) Orden de filas preservado respecto al raw
    # ------------------------------------------------------------
    same_order_train = _same_row_order(train_raw, train_scaled)
    same_order_valid = _same_row_order(valid_raw, valid_scaled)
    same_order_test = _same_row_order(test_raw, test_scaled)

    print("\nOrden de filas preservado respecto al raw:")
    print(f"  TRAIN: {same_order_train}")
    print(f"  VALID: {same_order_valid}")
    print(f"  TEST : {same_order_test}")

    # ------------------------------------------------------------
    # 7) Orden temporal del dataset escalado
    # ------------------------------------------------------------
    sorted_train = _is_sorted(train_scaled)
    sorted_valid = _is_sorted(valid_scaled)
    sorted_test = _is_sorted(test_scaled)

    print("\nScaled sigue ordenado temporalmente:")
    print(f"  TRAIN: {sorted_train}")
    print(f"  VALID: {sorted_valid}")
    print(f"  TEST : {sorted_test}")

    # ------------------------------------------------------------
    # 8) Índice (DatetimeIndex) preservado
    # ------------------------------------------------------------
    idx_train = _check_index_equal(train_raw, train_scaled)
    idx_valid = _check_index_equal(valid_raw, valid_scaled)
    idx_test = _check_index_equal(test_raw, test_scaled)

    print("\nÍndice (DatetimeIndex) sin modificar:")
    print(f"  TRAIN: {idx_train if idx_train is not None else 'N/A'}")
    print(f"  VALID: {idx_valid if idx_valid is not None else 'N/A'}")
    print(f"  TEST : {idx_test if idx_test is not None else 'N/A'}")

    # ------------------------------------------------------------
    # 9) Summary estructurado
    # ------------------------------------------------------------
    summary = {
        "name": name,
        "scale_cols": scale_cols,
        "n_scale_cols": len(scale_cols),

        "train_standardization": {
            "tol_mean": tol_mean,
            "tol_std": tol_std,
            "n_mean_fail": len(mean_fail_cols),
            "n_std_fail": len(std_fail_cols),
            "mean_fail_cols": mean_fail_cols,
            "std_fail_cols": std_fail_cols,
            "mean_ok": len(mean_fail_cols) == 0,
            "std_ok": len(std_fail_cols) == 0,
        },

        "transformation_applied": {
            "valid": valid_transform_report,
            "test": test_transform_report,
        },

        "target_unchanged": {
            "train": target_same_train,
            "valid": target_same_valid,
            "test": target_same_test,
        },

        "date_unchanged": {
            "train": date_same_train,
            "valid": date_same_valid,
            "test": date_same_test,
        },

        "row_order_preserved_vs_raw": {
            "train": same_order_train,
            "valid": same_order_valid,
            "test": same_order_test,
        },

        "scaled_temporal_order_ok": {
            "train": sorted_train,
            "valid": sorted_valid,
            "test": sorted_test,
        },

        "datetime_index_preserved": {
            "train": idx_train,
            "valid": idx_valid,
            "test": idx_test,
        },
    }

    # ------------------------------------------------------------
    # 10) Diagnóstico final
    # ------------------------------------------------------------
    all_ok = (
        summary["train_standardization"]["mean_ok"]
        and summary["train_standardization"]["std_ok"]
        and all(summary["target_unchanged"].values())
        and all(summary["date_unchanged"].values())
        and all(summary["row_order_preserved_vs_raw"].values())
        and all(summary["scaled_temporal_order_ok"].values())
        and all(v is True for v in summary["datetime_index_preserved"].values() if v is not None)
    )

    summary["all_checks_passed"] = all_ok

    print("\nDiagnóstico final:")
    print(f"  all_checks_passed: {all_ok}")
    print("=" * 60)

    return summary

In [69]:
targets_col

['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']

In [70]:
scaling_check_target_0 = verify_scaling(
    train_raw=mnq_t2_train,
    train_scaled=mnq_t2_train_scaled,
    valid_raw=mnq_t2_valid,
    valid_scaled=mnq_t2_valid_scaled,
    test_raw=mnq_t2_test,
    test_scaled=mnq_t2_test_scaled,
    name="MNQ T2 R2 Scaling Check",
    target_col=targets_col[0],
    features_to_scale=features_to_scale,
    date_col="date",
    minute_col="minute_of_day",
    tol_mean=0.05,
    tol_std=0.05,
)

scaling_check_target_1 = verify_scaling(
    train_raw=mnq_t2_train,
    train_scaled=mnq_t2_train_scaled,
    valid_raw=mnq_t2_valid,
    valid_scaled=mnq_t2_valid_scaled,
    test_raw=mnq_t2_test,
    test_scaled=mnq_t2_test_scaled,
    name="MNQ T2 R2 Scaling Check",
    target_col=targets_col[1],
    features_to_scale=features_to_scale,
    date_col="date",
    minute_col="minute_of_day",
    tol_mean=0.05,
    tol_std=0.05,
)


scaling_check_target_2 = verify_scaling(
    train_raw=mnq_t2_train,
    train_scaled=mnq_t2_train_scaled,
    valid_raw=mnq_t2_valid,
    valid_scaled=mnq_t2_valid_scaled,
    test_raw=mnq_t2_test,
    test_scaled=mnq_t2_test_scaled,
    name="MNQ T2 R2 Scaling Check",
    target_col=targets_col[2],
    features_to_scale=features_to_scale,
    date_col="date",
    minute_col="minute_of_day",
    tol_mean=0.05,
    tol_std=0.05,
)




========================= MNQ T2 R2 Scaling Check =========================
Features a verificar (presentes): 7
['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd']

TRAIN (estandarización):
  Mean fuera tolerancia (|mean| > 0.05): 0
  Std  fuera tolerancia (|std-1| > 0.05): 0

Transformación aplicada:
  VALID - columnas modificadas: 7 / 7
  TEST  - columnas modificadas: 7 / 7

Target sin modificar:
  TRAIN: True
  VALID: True
  TEST : True

Date sin modificar:
  TRAIN: True
  VALID: True
  TEST : True

Orden de filas preservado respecto al raw:
  TRAIN: True
  VALID: True
  TEST : True

Scaled sigue ordenado temporalmente:
  TRAIN: True
  VALID: True
  TEST : True

Índice (DatetimeIndex) sin modificar:
  TRAIN: True
  VALID: True
  TEST : True

Diagnóstico final:
  all_checks_passed: True

========================= MNQ T2 R2 Scaling Check =========================
Features a verificar (presentes): 7
['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_no

### **Conclusión del paso de escalado**



El proceso de escalado ha sido implementado y validado correctamente en los tres conjuntos de datos (train, valid y test), cumpliendo con todos los criterios técnicos y temporales definidos para el pipeline.

En el conjunto de entrenamiento, las variables escaladas presentan media aproximadamente cero y desviación estándar cercana a uno, sin desviaciones fuera de los umbrales establecidos. Esto confirma que el ajuste del scaler se realizó de forma correcta y exclusivamente sobre el conjunto de entrenamiento.

En los conjuntos de validación y prueba, todas las variables definidas fueron transformadas, lo que garantiza que el mismo scaler fue aplicado de forma consistente y que no existe reutilización de datos sin procesar ni filtrado de información futura.

Las variables críticas del problema se mantienen intactas. En particular, la variable objetivo no fue modificada y la columna de fecha conserva exactamente sus valores originales, preservando la semántica del problema y evitando alteraciones en la interpretación de los datos.

Desde el punto de vista temporal, el escalado no introduce ningún tipo de distorsión. El orden de las filas se mantiene idéntico respecto al dataset original y, adicionalmente, los datasets escalados continúan correctamente ordenados por fecha y minuto intradía. Esto garantiza que la estructura secuencial del problema se conserva completamente.

El índice temporal (DatetimeIndex) también se preserva sin modificaciones, lo cual es fundamental para etapas posteriores como la generación de ventanas, operaciones de alineación temporal y procesos de backtesting.

Los resultados son consistentes entre los distintos targets evaluados, lo cual es esperable dado que el escalado depende únicamente de las features y no de la variable objetivo.

En conjunto, se verifica que no existe data leakage, no se ha alterado la estructura temporal del dataset y el proceso de transformación se ha aplicado de manera correcta y reproducible.

Se concluye que el dataset escalado se encuentra en condiciones adecuadas para continuar con la siguiente etapa del pipeline, correspondiente a la generación de ventanas secuenciales (seq2one) utilizando un tamaño de ventana de 30.


# **4. Generación de ventanas deslizantes (sliding windows)**

## **4.1. Introducción conceptual**

La generación de ventanas constituye una etapa fundamental en la preparación de los datos para el modelado, ya que permite transformar el dataset tabular en una representación secuencial adecuada para capturar dependencias temporales en el comportamiento intradía del mercado.

En este proyecto se adopta un enfoque **seq2one**, en el cual cada muestra está compuesta por una secuencia de observaciones pasadas (ventana) y un único valor objetivo asociado.

Cada ventana de entrada $ X_i$  está formada por una secuencia de longitud fija  $L$, construida a partir de las features en filas consecutivas del dataset:

$$
X_i = [x_i, x_{i+1}, ..., x_{i+L-1}]
$$

La salida asociada $ y_i $ corresponde al valor del target en la **última fila de la ventana**:

$$
y_i = y_{i+L-1}
$$

En esta etapa, los targets utilizados son:

* `t2_p40_h30`
* `t2_p40_h60`
* `t2_p50_h30`

Estos targets han sido construidos previamente incorporando un horizonte hacia adelante (retorno futuro con umbral), por lo que no es necesario aplicar desplazamientos adicionales durante la generación de ventanas. En consecuencia, la relación entre las ventanas y el target es directa y no depende explícitamente del horizonte en esta etapa.

Dado un día con $ N $ observaciones, el número de ventanas generadas es:

$$
N - L + 1
$$

En este proyecto, el análisis estructural del dataset muestra que cada jornada intradía contiene exactamente 60 observaciones, correspondientes al régimen de apertura (opening). Esto implica que el tamaño de ventana $ L $ está limitado por la longitud disponible por sesión.

La generación de ventanas se realiza de forma independiente por cada jornada intradía, evitando la mezcla de información entre días distintos. Esto es crítico para preservar la estructura del mercado y evitar cualquier forma de *data leakage* entre sesiones.

Cada muestra generada tiene la siguiente estructura:

* Entrada: matriz de dimensión $ L \times F $, donde $ F $ es el número de features.
* Salida: un valor escalar correspondiente al target.

El tamaño de la ventana $ L $ es un hiperparámetro clave que controla la cantidad de información histórica disponible para el modelo. En este caso, debido a la longitud fija de las sesiones (60 observaciones), se adopta:

$$
L = 30
$$

Esta elección permite maximizar la cantidad de muestras disponibles por día ($ 31 $ ventanas por sesión), manteniendo suficiente contexto temporal reciente para el modelado. Ventanas mayores resultan subóptimas o inviables bajo esta restricción estructural.

Es importante destacar que, aunque la ventana es de longitud 30, varias features incorporan memoria histórica de mayor alcance (por ejemplo, indicadores con lookback de 60 minutos). Esto permite que el modelo disponga simultáneamente de:

* contexto temporal explícito (ventana)
* memoria histórica implícita (features)

Finalmente, la generación de ventanas se realiza sobre los datasets previamente escalados, asegurando que todas las features se encuentren en una escala homogénea antes de ser utilizadas por los modelos. Esto facilita el entrenamiento y mejora la estabilidad numérica en etapas posteriores del pipeline.


## **4.2. Construcción de ventanas `seq2one`**

### **4.2.1. Generador seq2one**

In [71]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from typing import List, Tuple, Dict, Any


def generate_windows_seq2one(
    df: pd.DataFrame,
    *,
    date_col: str,
    features: List[str],
    target_col: str,
    window_size: int,
    minute_col: str = "minute_of_day",
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
    validate_temporal_order: bool = True,
    validate_unique_minutes: bool = True,
    return_summary: bool = False,
) -> Tuple[np.ndarray, np.ndarray] | Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    """
    Genera ventanas seq2one a partir de un DataFrame intradía.

    Esquema
    -------
    Para cada día:
      - X_i = [x_i, ..., x_{i+L-1}]
      - y_i = target en la última fila de la ventana

    Esto implica:
      y_i = y[i + window_size - 1]

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset intradía ya escalado.
    date_col : str
        Columna que identifica la sesión/día.
    features : List[str]
        Features de entrada.
    target_col : str
        Columna target.
    window_size : int
        Longitud L de la ventana.
    minute_col : str
        Columna de orden intradía.
    flatten : bool
        Si True, retorna X con shape (N, L*F).
    drop_windows_with_nan : bool
        Si True, descarta ventanas con NaNs en X o y.
    validate_temporal_order : bool
        Si True, exige orden ascendente por minuto dentro de cada día.
    validate_unique_minutes : bool
        Si True, exige unicidad de minute_col dentro de cada día.
    return_summary : bool
        Si True, retorna además un resumen estructurado.

    Retorna
    -------
    X : np.ndarray
        - flatten=False -> shape (N, L, F), dtype float32
        - flatten=True  -> shape (N, L*F), dtype float32
    y : np.ndarray
        shape (N,), dtype int8 para targets T2
    """

    # ------------------------------------------------------------
    # 1) Validaciones básicas
    # ------------------------------------------------------------
    if window_size <= 0:
        raise ValueError("window_size debe ser un entero positivo.")

    required_cols = [date_col] + features + [target_col]
    if validate_temporal_order or validate_unique_minutes:
        required_cols.append(minute_col)

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en df: {missing}")

    # ------------------------------------------------------------
    # 2) Acumuladores
    # ------------------------------------------------------------
    X_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []
    F = len(features)

    n_days_total = 0
    n_days_used = 0
    n_days_skipped_short = 0
    n_windows_total = 0

    # ------------------------------------------------------------
    # 3) Generación por día
    # ------------------------------------------------------------
    for day_value, g in df.groupby(date_col, sort=False):
        n_days_total += 1
        g = g.reset_index(drop=True)
        n = len(g)

        # --------------------------------------------------------
        # 3.1) Validaciones dentro del día
        # --------------------------------------------------------
        if validate_temporal_order and not g[minute_col].is_monotonic_increasing:
            raise ValueError(
                f"Desorden temporal detectado en {date_col}={day_value}: "
                f"'{minute_col}' no está en orden ascendente."
            )

        if validate_unique_minutes:
            duplicated_mask = g[minute_col].duplicated(keep=False)
            if duplicated_mask.any():
                duplicated_values = g.loc[duplicated_mask, minute_col].tolist()[:10]
                raise ValueError(
                    f"Minutos duplicados detectados en {date_col}={day_value}: "
                    f"{duplicated_values}"
                )

        # Si no alcanza para una ventana completa
        if n < window_size:
            n_days_skipped_short += 1
            continue

        # --------------------------------------------------------
        # 3.2) Conversión a numpy
        # --------------------------------------------------------
        Xg = g[features].to_numpy(dtype=np.float32, copy=False)
        yg = g[target_col].to_numpy(copy=False)

        # --------------------------------------------------------
        # 3.3) Ventanas deslizantes sobre X
        # --------------------------------------------------------
        Xw = sliding_window_view(Xg, window_shape=window_size, axis=0)

        # Normalizar a shape (N, L, F)
        if Xw.ndim != 3:
            raise ValueError(f"Xw ndim inesperado: {Xw.ndim} | shape={Xw.shape}")

        if Xw.shape[1] == F and Xw.shape[2] == window_size:
            Xw = np.swapaxes(Xw, 1, 2)

        if Xw.shape[1] != window_size or Xw.shape[2] != F:
            raise ValueError(
                f"Xw shape inválido tras normalizar: {Xw.shape}. "
                f"Esperado: (N, {window_size}, {F})"
            )

        # --------------------------------------------------------
        # 3.4) Target alineado al final de la ventana
        # --------------------------------------------------------
        yw = yg[window_size - 1:]

        # --------------------------------------------------------
        # 3.5) Filtrado de NaNs
        # --------------------------------------------------------
        if drop_windows_with_nan:
            x_ok = ~np.isnan(Xw).any(axis=(1, 2))
            y_ok = ~pd.isna(yw)

            ok = x_ok & y_ok
            Xw = Xw[ok]
            yw = yw[ok]

        if Xw.size == 0:
            continue

        n_days_used += 1
        n_windows_total += Xw.shape[0]

        # --------------------------------------------------------
        # 3.6) Aplanado opcional
        # --------------------------------------------------------
        if flatten:
            Xw = Xw.reshape(Xw.shape[0], -1)

        # --------------------------------------------------------
        # 3.7) Cast final
        # --------------------------------------------------------
        Xw = Xw.astype(np.float32, copy=False)
        yw = np.asarray(yw, dtype=np.int8)

        X_all.append(Xw)
        y_all.append(yw)

    # ------------------------------------------------------------
    # 4) Caso borde: no se generaron ventanas
    # ------------------------------------------------------------
    if not X_all:
        X = np.empty((0, window_size, F), dtype=np.float32)
        if flatten:
            X = X.reshape(0, window_size * F)

        y = np.empty((0,), dtype=np.int8)

        summary = {
            "target_col": target_col,
            "window_size": window_size,
            "n_days_total": n_days_total,
            "n_days_used": 0,
            "n_days_skipped_short": n_days_skipped_short,
            "n_windows_total": 0,
            "n_features": F,
            "flatten": flatten,
        }

        return (X, y, summary) if return_summary else (X, y)

    # ------------------------------------------------------------
    # 5) Concatenación final
    # ------------------------------------------------------------
    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)

    summary = {
        "target_col": target_col,
        "window_size": window_size,
        "n_days_total": n_days_total,
        "n_days_used": n_days_used,
        "n_days_skipped_short": n_days_skipped_short,
        "n_windows_total": int(X.shape[0]),
        "n_features": F,
        "flatten": flatten,
        "X_shape": list(X.shape),
        "y_shape": list(y.shape),
    }

    return (X, y, summary) if return_summary else (X, y)

### **4.2.2. Load/Build para SEQ2ONE**

In [72]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Tuple


def prepare_or_load_seq2one_windows_npz(
    *,
    mnq_train,
    mnq_valid,
    mnq_test,
    features: List[str],
    target_col: str,
    window_size: int,
    out_train,
    out_valid,
    out_test,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
    verbose: bool = True,
    repair_axis_order_if_needed: bool = True,
    validate_temporal_order: bool = True,
    validate_unique_minutes: bool = True,
    out_meta_train=None,
    out_meta_valid=None,
    out_meta_test=None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Prepara o carga ventanas seq2one desde disco en formato .npz.

    Flujo
    -----
    Para cada split:
      - Si existe el .npz, lo carga.
      - Si no existe, construye ventanas con generate_windows_seq2one(...) y las guarda.
      - Opcionalmente repara el orden de ejes si detecta (N, F, L).
      - Opcionalmente guarda metadata por split.

    Supuesto de uso
    ---------------
    Esta función se aplica sobre datasets ya escalados y listos para modelado.

    Retorna
    -------
    (X_train, y_train, X_valid, y_valid, X_test, y_test)
    """

    # ------------------------------------------------------------
    # 1) Helpers de path
    # ------------------------------------------------------------
    def _to_path(p) -> Path | None:
        if p is None:
            return None
        return p if isinstance(p, Path) else Path(str(p))

    def _ensure_parent_dir(path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)

    out_train = _to_path(out_train)
    out_valid = _to_path(out_valid)
    out_test = _to_path(out_test)

    out_meta_train = _to_path(out_meta_train)
    out_meta_valid = _to_path(out_meta_valid)
    out_meta_test = _to_path(out_meta_test)

    if out_train is None or out_valid is None or out_test is None:
        raise ValueError("out_train, out_valid y out_test son obligatorios.")

    F = len(features)

    # ------------------------------------------------------------
    # 2) Helper para verificar orden temporal del DataFrame
    # ------------------------------------------------------------
    def _check_temporal_order(df, split_name: str) -> None:
        if not validate_temporal_order:
            return

        required = [date_col, minute_col]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(
                f"[{split_name}] faltan columnas para validar orden temporal: {missing}"
            )

        ordered = (
            df[[date_col, minute_col]]
            .reset_index(drop=True)
            .equals(
                df[[date_col, minute_col]]
                .sort_values([date_col, minute_col], kind="mergesort")
                .reset_index(drop=True)
            )
        )

        if not ordered:
            raise ValueError(
                f"[{split_name}] el DataFrame no está ordenado por "
                f"[{date_col}, {minute_col}] antes de generar ventanas."
            )

    # ------------------------------------------------------------
    # 3) Reparación opcional del orden de ejes en X
    # ------------------------------------------------------------
    def _maybe_repair_X(X: np.ndarray) -> np.ndarray:
        if flatten or not repair_axis_order_if_needed:
            return X

        if X.ndim == 3 and X.shape[1] == F and X.shape[2] == window_size:
            return np.swapaxes(X, 1, 2)  # (N,F,L) -> (N,L,F)

        return X

    # ------------------------------------------------------------
    # 4) Guardado / carga NPZ
    # ------------------------------------------------------------
    def _save_npz(path_npz: Path, X: np.ndarray, y: np.ndarray) -> None:
        _ensure_parent_dir(path_npz)

        X_to_save = np.asarray(X, dtype=np.float32)
        y_to_save = np.asarray(y)

        np.savez_compressed(path_npz, X=X_to_save, y=y_to_save)

    def _load_npz(path_npz: Path):
        with np.load(path_npz, allow_pickle=False) as z:
            X = z["X"]
            y = z["y"]
        return X, y

    # ------------------------------------------------------------
    # 5) Guardado de metadata opcional
    # ------------------------------------------------------------
    def _save_meta(
        path_meta: Path | None,
        split_name: str,
        X: np.ndarray,
        y: np.ndarray,
        df_source,
    ) -> None:
        if path_meta is None:
            return

        _ensure_parent_dir(path_meta)

        meta = {
            "split": split_name,
            "target_col": target_col,
            "features": list(features),
            "n_features": len(features),
            "window_size": window_size,
            "flatten": flatten,
            "drop_windows_with_nan": drop_windows_with_nan,
            "validate_temporal_order": validate_temporal_order,
            "validate_unique_minutes": validate_unique_minutes,
            "date_col": date_col,
            "minute_col": minute_col,
            "n_days": int(df_source[date_col].nunique()),
            "n_rows_input": int(len(df_source)),
            "n_samples": int(len(X)),
            "X_shape": list(X.shape),
            "y_shape": list(y.shape),
            "X_dtype": str(X.dtype),
            "y_dtype": str(y.dtype),
            "sorted_by": [date_col, minute_col],
        }

        with path_meta.open("w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2, ensure_ascii=False)

    # ------------------------------------------------------------
    # 6) Lógica principal por split
    # ------------------------------------------------------------
    def _load_or_build(split_name: str, df, path_npz: Path, path_meta: Path | None):
        action = "load"

        if not path_npz.exists():
            action = "build"

            _check_temporal_order(df, split_name)

            X, y = generate_windows_seq2one(
                df=df,
                date_col=date_col,
                features=features,
                target_col=target_col,
                window_size=window_size,
                minute_col=minute_col,
                flatten=flatten,
                drop_windows_with_nan=drop_windows_with_nan,
                validate_temporal_order=validate_temporal_order,
                validate_unique_minutes=validate_unique_minutes,
            )

            _save_npz(path_npz, X, y)
            _save_meta(path_meta, split_name, X, y, df)

        else:
            X, y = _load_npz(path_npz)

        # --------------------------------------------------------
        # Reparación opcional de ejes
        # --------------------------------------------------------
        X2 = _maybe_repair_X(X)

        if (X2 is not X) and action == "load":
            X2_np = np.asarray(X2, dtype=np.float32)
            y_np = np.asarray(y)
            _save_npz(path_npz, X2_np, y_np)
            _save_meta(path_meta, split_name, X2_np, y_np, df)
            X = X2_np
            y = y_np
        else:
            X = X2

        if verbose:
            print(
                f"[{target_col} | L={window_size} | {split_name}] {action.upper()}  "
                f"X={X.shape}  y={y.shape}  -> {path_npz.name}"
            )

        return X, y

    # ------------------------------------------------------------
    # 7) Ejecutar para train / valid / test
    # ------------------------------------------------------------
    X_train, y_train = _load_or_build("train", mnq_train, out_train, out_meta_train)
    X_valid, y_valid = _load_or_build("valid", mnq_valid, out_valid, out_meta_valid)
    X_test, y_test = _load_or_build("test", mnq_test, out_test, out_meta_test)

    return X_train, y_train, X_valid, y_valid, X_test, y_test

### **4.2.3. Info para SEQ2ONE**

In [73]:
from pathlib import Path
import numpy as np
import pandas as pd


def xy_info_seq2one_compact(
    target_col: str,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    *,
    window_size: int,
    n_features: int,
    flatten: bool = False,
    path_train: str | Path | None = None,
    path_valid: str | Path | None = None,
    path_test:  str | Path | None = None,
    return_summary: bool = False,
):
    """
    Muestra información estructural compacta de ventanas seq2one por split.

    Reporta:
    - shape de X e y
    - dtype de X e y
    - validación de dimensiones esperadas
    - presencia de NaN
    - tamaño estimado en RAM
    - tamaño en disco del .npz (si se provee path)
    - clases únicas de y
    """

    # ------------------------------------------------------------
    # 1) Helpers
    # ------------------------------------------------------------
    def _to_path(p):
        return None if p is None else (p if isinstance(p, Path) else Path(str(p)))

    def _human_bytes(n: int) -> str:
        units = ["B", "KB", "MB", "GB", "TB"]
        x = float(n)
        for u in units:
            if x < 1024.0 or u == units[-1]:
                return f"{x:.2f}{u}"
            x /= 1024.0
        return f"{x:.2f}TB"

    def _file_info(path: Path | None, X: np.ndarray, y: np.ndarray) -> dict:
        if path is None or not path.exists():
            return {
                "exists": False,
                "file_size_bytes": None,
                "file_size_human": "NA",
                "raw_estimated_bytes": int(getattr(X, "nbytes", 0) + getattr(y, "nbytes", 0)),
                "compression_ratio": None,
                "kind": "file=NA",
            }

        size_disk = path.stat().st_size
        raw_est = int(getattr(X, "nbytes", 0) + getattr(y, "nbytes", 0))
        ratio = (size_disk / raw_est) if raw_est > 0 else float("nan")

        if raw_est > 0 and ratio < 0.85:
            kind = "npz-compressed (probable)"
        else:
            kind = "npz (sin compresión o compresión mínima)"

        return {
            "exists": True,
            "file_size_bytes": int(size_disk),
            "file_size_human": _human_bytes(size_disk),
            "raw_estimated_bytes": raw_est,
            "raw_estimated_human": _human_bytes(raw_est),
            "compression_ratio": float(ratio) if not np.isnan(ratio) else None,
            "kind": kind,
        }

    def _nan_count(arr) -> int:
        try:
            return int(np.isnan(arr).sum())
        except Exception:
            try:
                return int(pd.isna(arr).sum())
            except Exception:
                return -1

    def _unique_y_info(y):
        try:
            vals = np.unique(y)
            if len(vals) <= 10:
                return vals.tolist()
            return f"{len(vals)} valores únicos"
        except Exception:
            return "NA"

    # ------------------------------------------------------------
    # 2) Paths opcionales
    # ------------------------------------------------------------
    p_train = _to_path(path_train)
    p_valid = _to_path(path_valid)
    p_test  = _to_path(path_test)

    summaries = {}

    # ------------------------------------------------------------
    # 3) Impresión por split
    # ------------------------------------------------------------
    def _analyze_split(name, X, y, path: Path | None):
        if not hasattr(X, "shape") or not hasattr(y, "shape"):
            print(f"{name} | ERROR: X o y sin shape")
            return {
                "status": "ERROR",
                "reason": "X o y sin shape",
            }

        N = X.shape[0] if X.ndim >= 1 else 0

        # --------------------------------------------------------
        # Validación de dimensiones esperadas
        # --------------------------------------------------------
        if flatten:
            if X.ndim == 2:
                dim = X.shape[1]
                expected = window_size * n_features
                x_ok = (dim == expected)
                dim_info = f"dim={dim}"
            else:
                x_ok = False
                dim_info = "dim=?"
        else:
            if X.ndim == 3:
                L, F = X.shape[1], X.shape[2]
                x_ok = (L == window_size) and (F == n_features)
                dim_info = f"(L,F)=({L},{F})"
            else:
                x_ok = False
                dim_info = "dim=?"

        # --------------------------------------------------------
        # Validación de y
        # --------------------------------------------------------
        y_ok = (
            (y.ndim == 1 and y.shape[0] == N) or
            (y.ndim == 2 and y.shape == (N, 1))
        )

        status = "OK" if x_ok and y_ok else "ERROR"

        # --------------------------------------------------------
        # dtype / NaN / memoria
        # --------------------------------------------------------
        x_dtype = str(getattr(X, "dtype", "NA"))
        y_dtype = str(getattr(y, "dtype", "NA"))
        x_mem_bytes = int(getattr(X, "nbytes", 0))
        y_mem_bytes = int(getattr(y, "nbytes", 0))
        x_mem = _human_bytes(x_mem_bytes)
        y_mem = _human_bytes(y_mem_bytes)
        x_nan = _nan_count(X)
        y_nan = _nan_count(y)
        y_classes = _unique_y_info(y)

        # --------------------------------------------------------
        # Tamaño archivo
        # --------------------------------------------------------
        finfo = _file_info(path, X, y)

        print(
            f"{name:<35} | "
            f"X:{X.shape} ({x_dtype}) | "
            f"y:{y.shape} ({y_dtype}) | "
            f"N={N} | "
            f"{dim_info} | "
            f"{status} | "
            f"RAM X={x_mem} y={y_mem} | "
            f"NaN X={x_nan} y={y_nan} | "
            f"classes={y_classes} | "
            f"file={finfo['file_size_human'] if finfo['exists'] else 'NA'}"
        )

        return {
            "status": status,
            "X_shape": list(X.shape),
            "y_shape": list(y.shape),
            "X_dtype": x_dtype,
            "y_dtype": y_dtype,
            "n_samples": int(N),
            "flatten": flatten,
            "window_size": int(window_size),
            "n_features": int(n_features),
            "x_ok": bool(x_ok),
            "y_ok": bool(y_ok),
            "dim_info": dim_info,
            "X_nan_count": int(x_nan),
            "y_nan_count": int(y_nan),
            "X_mem_bytes": x_mem_bytes,
            "y_mem_bytes": y_mem_bytes,
            "X_mem_human": x_mem,
            "y_mem_human": y_mem,
            "y_classes": y_classes,
            "file_info": finfo,
        }

    # ------------------------------------------------------------
    # 4) Salida final por split
    # ------------------------------------------------------------
    summaries["train"] = _analyze_split(
        f"[{target_col} | L={window_size} | train]", X_train, y_train, p_train
    )
    summaries["valid"] = _analyze_split(
        f"[{target_col} | L={window_size} | valid]", X_valid, y_valid, p_valid
    )
    summaries["test"] = _analyze_split(
        f"[{target_col} | L={window_size} | test ]", X_test, y_test, p_test
    )
    print()

    summaries["target_col"] = target_col
    summaries["all_ok"] = all(
        summaries[s]["status"] == "OK" for s in ["train", "valid", "test"]
    )

    if return_summary:
        return summaries

### **4.2.4. Utilidades**

In [74]:
# ============================================================
# Features para ventanas
# ============================================================
features_to_windows = features_to_scale.copy()

n_features = len(features_to_windows)

print(f'features_to_windows ({n_features}): {features_to_windows}')
print(f'targets_col: {targets_col}')

# ============================================================
# Window size (definido por análisis previo)
# ============================================================
window_sizes = [30]

print(f'window_sizes: {window_sizes}')


# ============================================================
# Mapa target -> datasets escalados
# ============================================================
targets_windows = {
    target: (mnq_t2_train_scaled, mnq_t2_valid_scaled, mnq_t2_test_scaled)
    for target in targets_col
}


# ============================================================
# Directorio de salida
# ============================================================
OUT_WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / "data/07_windows/seq2one"

features_to_windows (7): ['ema_60', 'roc_60', 'roc_30', 'stoch_k_30', 'mom_5', 'atr_norm_10', 'macd']
targets_col: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
window_sizes: [30]


### **4.2.5. Cargar ventanas .npZ**

In [75]:
def _load_npz(path_npz: Path):
    """
    Carga arrays X e y desde un archivo .npz.

    Validaciones:
    - Verifica que el archivo exista.
    - Verifica que el .npz contenga claves 'X' e 'y'.
    - Mantiene dtype consistente con lo guardado.
    """

    if not path_npz.exists():
        raise FileNotFoundError(f"No se encontró el archivo NPZ: {path_npz}")

    with np.load(path_npz, allow_pickle=False) as z:

        if "X" not in z:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path_npz}")

        if "y" not in z:
            raise KeyError(f"NPZ inválido (falta clave 'y'): {path_npz}")

        X = z["X"]
        y = z["y"]

    # Normalización mínima del pipeline
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y)

    return X, y

### **4.2.6. Generación de resumen de ventanas `seq2one`**

In [76]:
from pathlib import Path
import numpy as np


def _human_bytes(n: int | None) -> str | None:
    if n is None:
        return None

    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)

    for u in units:
        if x < 1024.0 or u == units[-1]:
            return f"{x:.2f}{u}"
        x /= 1024.0

    return f"{x:.2f}TB"


def _split_file_info(path: Path, X: np.ndarray, y: np.ndarray) -> dict:
    """
    Devuelve metadata estructural de un split de ventanas.

    Incluye:
    - tamaño real del .npz en disco
    - tamaño bruto estimado en RAM (X.nbytes + y.nbytes)
    - ratio de compresión aproximado
    - shapes y dtypes
    """
    path = Path(path)
    exists = path.exists()

    size_disk = path.stat().st_size if exists else None
    raw_est = int(getattr(X, "nbytes", 0) + getattr(y, "nbytes", 0))
    ratio = (size_disk / raw_est) if (size_disk is not None and raw_est > 0) else None

    if ratio is None:
        compression_kind = None
    else:
        compression_kind = (
            "compressed"
            if ratio < 0.85
            else "low_or_no_compression"
        )

    return {
        "file_exists": exists,
        "file_name": path.name,
        "file_path": str(path),
        "size_disk_bytes": size_disk,
        "size_disk_human": _human_bytes(size_disk),
        "raw_estimated_bytes": raw_est,
        "raw_estimated_human": _human_bytes(raw_est),
        "compression_ratio": round(ratio, 6) if ratio is not None else None,
        "compression_kind": compression_kind,
        "shape_X": list(getattr(X, "shape", ())),
        "shape_y": list(getattr(y, "shape", ())),
        "dtype_X": str(getattr(X, "dtype", "NA")),
        "dtype_y": str(getattr(y, "dtype", "NA")),
    }

In [77]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np


def update_seq2one_manifest(
    manifest: dict,
    *,
    target_col: str,
    window_size: int,
    n_features: int,
    features: list[str],
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test,
    path_train: Path,
    path_valid: Path,
    path_test: Path,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
) -> dict:
    """
    Actualiza el manifest global con la entrada (target_col, window_size).

    Retorna el manifest actualizado.
    La mutación también ocurre in-place sobre manifest.
    """

    # ------------------------------------------------------------
    # 1. Asegurar estructura base del manifest
    # ------------------------------------------------------------
    if "manifest_version" not in manifest:
        manifest["manifest_version"] = 1

    if "format" not in manifest:
        manifest["format"] = "seq2one_npz"

    if "items" not in manifest:
        manifest["items"] = {}

    # ------------------------------------------------------------
    # 2. Clave única por configuración
    # ------------------------------------------------------------
    key = f"{target_col}__L{window_size}"

    # ------------------------------------------------------------
    # 3. Normalizar arrays
    # ------------------------------------------------------------
    X_train_np = np.asarray(X_train)
    y_train_np = np.asarray(y_train)

    X_valid_np = np.asarray(X_valid)
    y_valid_np = np.asarray(y_valid)

    X_test_np = np.asarray(X_test)
    y_test_np = np.asarray(y_test)

    # ------------------------------------------------------------
    # 4. Actualizar manifest
    # ------------------------------------------------------------
    manifest["items"][key] = {
        "target": target_col,
        "window_size": int(window_size),
        "n_features": int(n_features),
        "features": list(features),
        "date_col": date_col,
        "minute_col": minute_col,
        "flatten": bool(flatten),
        "drop_windows_with_nan": bool(drop_windows_with_nan),
        "format_expected": "npz_compressed",
        "updated_at": datetime.utcnow().isoformat() + "Z",
        "splits": {
            "train": _split_file_info(Path(path_train), X_train_np, y_train_np),
            "valid": _split_file_info(Path(path_valid), X_valid_np, y_valid_np),
            "test": _split_file_info(Path(path_test), X_test_np, y_test_np),
        },
    }

    return manifest


def save_seq2one_manifest(manifest: dict, *, out_path: Path) -> Path:
    """
    Guarda el manifest global de ventanas seq2one en formato JSON.

    - Crea directorios si no existen.
    - Agrega timestamp de guardado.
    - Ordena claves para reproducibilidad.
    """

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Asegurar estructura mínima
    if "manifest_version" not in manifest:
        manifest["manifest_version"] = 1

    if "format" not in manifest:
        manifest["format"] = "seq2one_npz"

    if "items" not in manifest:
        manifest["items"] = {}

    # Timestamp global
    manifest["saved_at"] = datetime.utcnow().isoformat() + "Z"

    # Guardado
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(
            manifest,
            f,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )

    print(f"[MANIFEST] Guardado: {out_path}")

    return out_path

### **4.2.7. Generación de ventanas `seq2one`**

In [78]:
from pathlib import Path
from datetime import datetime

# ============================================================
# Manifest global
# ============================================================
manifest = {
    "schema": "seq2one_windows_manifest_v1",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "items": {}
}

# ============================================================
# Loop principal
# ============================================================
for target_col, (df_tr, df_va, df_te) in targets_windows.items():
    for L in window_sizes:

        print("\n" + "=" * 70)
        print(f"SEQ2ONE | TARGET={target_col} | L={L} | F={len(features_to_windows)}")
        print("=" * 70)

        # --------------------------------------------------------
        # Directorio por tamaño de ventana
        # --------------------------------------------------------
        out_dir_L = OUT_WINDOWS_SEQ2ONE_DIR / f"L{L}"
        out_dir_L.mkdir(parents=True, exist_ok=True)

        # --------------------------------------------------------
        # Archivos NPZ por split
        # --------------------------------------------------------
        out_train = out_dir_L / f"windows_{target_col}_train.npz"
        out_valid = out_dir_L / f"windows_{target_col}_valid.npz"
        out_test  = out_dir_L / f"windows_{target_col}_test.npz"

        # --------------------------------------------------------
        # Metadata opcional por split
        # --------------------------------------------------------
        out_meta_train = out_dir_L / f"meta_{target_col}_train.json"
        out_meta_valid = out_dir_L / f"meta_{target_col}_valid.json"
        out_meta_test  = out_dir_L / f"meta_{target_col}_test.json"

        # --------------------------------------------------------
        # Cargar o construir ventanas
        # --------------------------------------------------------
        X_train, y_train, X_valid, y_valid, X_test, y_test = prepare_or_load_seq2one_windows_npz(
            mnq_train=df_tr,
            mnq_valid=df_va,
            mnq_test=df_te,
            features=features_to_windows,
            target_col=target_col,
            window_size=L,
            out_train=out_train,
            out_valid=out_valid,
            out_test=out_test,
            out_meta_train=out_meta_train,
            out_meta_valid=out_meta_valid,
            out_meta_test=out_meta_test,
            date_col="date",
            minute_col="minute_of_day",
            flatten=False,
            drop_windows_with_nan=True,
            verbose=True,
            repair_axis_order_if_needed=True,
            validate_temporal_order=True,
            validate_unique_minutes=True,
        )

        # --------------------------------------------------------
        # Auditoría compacta de ventanas
        # --------------------------------------------------------
        xy_info_seq2one_compact(
            target_col=target_col,
            X_train=X_train,
            y_train=y_train,
            X_valid=X_valid,
            y_valid=y_valid,
            X_test=X_test,
            y_test=y_test,
            window_size=L,
            n_features=len(features_to_windows),
            flatten=False,
            path_train=out_train,
            path_valid=out_valid,
            path_test=out_test,
            return_summary=False,
        )

        # --------------------------------------------------------
        # Actualizar manifest global
        # --------------------------------------------------------
        manifest = update_seq2one_manifest(
            manifest=manifest,
            target_col=target_col,
            window_size=L,
            n_features=len(features_to_windows),
            features=features_to_windows,
            X_train=X_train,
            y_train=y_train,
            X_valid=X_valid,
            y_valid=y_valid,
            X_test=X_test,
            y_test=y_test,
            path_train=out_train,
            path_valid=out_valid,
            path_test=out_test,
            date_col="date",
            minute_col="minute_of_day",
            flatten=False,
            drop_windows_with_nan=True,
        )

# ============================================================
# Guardar manifest global
# ============================================================
save_seq2one_manifest(
    manifest=manifest,
    out_path=OUT_WINDOWS_SEQ2ONE_DIR / "seq2one_windows_manifest.json",
)


SEQ2ONE | TARGET=t2_p40_h30 | L=30 | F=7
[t2_p40_h30 | L=30 | train] BUILD  X=(32147, 30, 7)  y=(32147,)  -> windows_t2_p40_h30_train.npz
[t2_p40_h30 | L=30 | valid] BUILD  X=(6882, 30, 7)  y=(6882,)  -> windows_t2_p40_h30_valid.npz
[t2_p40_h30 | L=30 | test] BUILD  X=(6913, 30, 7)  y=(6913,)  -> windows_t2_p40_h30_test.npz
[t2_p40_h30 | L=30 | train]         | X:(32147, 30, 7) (float32) | y:(32147,) (int8) | N=32147 | (L,F)=(30,7) | OK | RAM X=25.75MB y=31.39KB | NaN X=0 y=0 | classes=[-1, 0, 1] | file=1.77MB
[t2_p40_h30 | L=30 | valid]         | X:(6882, 30, 7) (float32) | y:(6882,) (int8) | N=6882 | (L,F)=(30,7) | OK | RAM X=5.51MB y=6.72KB | NaN X=0 y=0 | classes=[-1, 0, 1] | file=389.67KB
[t2_p40_h30 | L=30 | test ]         | X:(6913, 30, 7) (float32) | y:(6913,) (int8) | N=6913 | (L,F)=(30,7) | OK | RAM X=5.54MB y=6.75KB | NaN X=0 y=0 | classes=[-1, 0, 1] | file=391.89KB


SEQ2ONE | TARGET=t2_p40_h60 | L=30 | F=7
[t2_p40_h60 | L=30 | train] BUILD  X=(32147, 30, 7)  y=(32147,)  -

PosixPath('/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/seq2one_windows_manifest.json')

**1. La generación de ventanas fue exitosa**

Para los tres targets, los archivos fueron construidos correctamente y el manifest global fue guardado sin errores.

**2. La estructura de las ventanas es consistente**

En todos los casos se obtuvo:

* `X` con shape `(N, 30, 8)`
* `y` con shape `(N,)`

Esto confirma que el esquema **seq2one** quedó correctamente implementado con:

* ventana de longitud 30
* 8 features por muestra
* target alineado al final de la ventana

**3. No hay problemas de integridad**

Todos los checks estructurales dieron correctamente:

* `dtype X = float32`
* `dtype y = int8`
* `NaN X = 0`
* `NaN y = 0`
* clases presentes: `[-1, 0, 1]`

Esto indica que las ventanas quedaron limpias, consistentes y listas para modelado.

**4. El número de muestras es razonable y estable entre targets**

Se generaron exactamente:

* train: `32,147`
* valid: `6,882`
* test: `6,913`

Esto es esperable porque:

* el dataset base por split es el mismo
* cambia solo el target
* la ventana también es la misma (`L = 30`)

Por lo tanto, la cantidad de muestras se mantiene constante entre targets.

**5. El tamaño en memoria y en disco es eficiente**

Los arrays ocupan en RAM aproximadamente:

* train: `29.43 MB`
* valid: `6.30 MB`
* test: `6.33 MB`

Y en disco los `.npz` ocupan bastante menos:

* train: ~`1.94 MB`
* valid/test: ~`427–429 KB`

Esto confirma que el almacenamiento comprimido está funcionando bien y que el volumen de datos es manejable para los experimentos posteriores.

**6. La trazabilidad del pipeline quedó bien resuelta**

Además de los archivos `.npz`, se guardó correctamente el manifest global en:

`data/07_windows/seq2one/seq2one_windows_manifest.json`

Esto es importante porque deja registro formal de:

* target
* tamaño de ventana
* features utilizadas
* shapes por split
* archivos generados

**7. Conclusión del paso**

La etapa de generación de ventanas quedó correctamente resuelta. Las tres configuraciones target fueron transformadas a formato secuencial sin errores, manteniendo consistencia estructural, ausencia de NaNs, tipos correctos y trazabilidad completa.

En consecuencia, los datasets están en condiciones adecuadas para avanzar a la siguiente etapa del pipeline: entrenamiento y evaluación de modelos sobre ventanas seq2one.


# **5. Alineamiento con libro de ML**

**Preprocesamiento y escalamiento de datos**

Este paso se realiza **después del split temporal y antes de la generación de ventanas**, siguiendo las buenas prácticas para modelado de series temporales financieras.

---

**Aspectos correctamente alineados**

1. Orden del proceso

* El split temporal se realiza antes de cualquier transformación.
* El escalamiento se aplica sobre los datasets tabulares.
* La generación de ventanas se realiza posteriormente sobre los datos ya escalados.

2. Regla crítica anti-leakage

* El scaler se ajusta exclusivamente con el conjunto de entrenamiento (TRAIN).
* Los conjuntos de validación y test se transforman utilizando ese mismo scaler, sin volver a ajustarlo.

3. Escalamiento aplicado únicamente a features

* Solo se escalan las variables de entrada seleccionadas (`roc_30`, `roc_60`, `stoch_k_30`, `atr_norm_10`).
* Variables categóricas como `regime_id` no se escalan.
* El target (`t2_dir_thr_90`, `t2_dir_thr_120`) no se transforma, preservando su naturaleza discreta.

4. Consistencia entre targets

* El mismo scaler se aplica independientemente del target utilizado.
* No se mezclan transformaciones entre configuraciones, garantizando comparabilidad entre experimentos.

5. Persistencia

* El scaler es guardado y reutilizado en todo el pipeline.
* Se garantiza reproducibilidad entre entrenamiento, validación, test e inferencia.

---

**Criterio de escalamiento**

Se utiliza StandardScaler (z-score), ajustado exclusivamente con el conjunto de entrenamiento.

El escalamiento se realiza por feature de forma global (no por día), permitiendo capturar la distribución completa del dataset sin introducir leakage temporal.

---

**Verificación automática del escalamiento**

Se realizó una validación sobre los datasets escalados, verificando que:

* la media por feature en TRAIN sea aproximadamente 0,
* la desviación estándar por feature en TRAIN sea aproximadamente 1,
* VALID y TEST hayan sido correctamente transformados,
* el target no haya sido modificado,
* la estructura temporal (orden, índice y alineación) se mantenga intacta.

Los resultados confirman que el escalamiento fue aplicado correctamente, sin introducir data leakage y preservando la integridad temporal del dataset.

---

**Conclusión**

El proceso de preprocesamiento y escalamiento se encuentra correctamente implementado y alineado con las buenas prácticas del libro.

Se garantiza que los datos utilizados para el modelado:

* respetan la causalidad temporal,
* mantienen consistencia entre splits,
* y se encuentran en una escala adecuada para el entrenamiento de modelos.


# **Conclusión global del stage**



1. Definición de targets

   Se definieron targets de tipo T2 (`t2_dir_thr_90` y `t2_dir_thr_120`) que incorporan un criterio de umbral y reducen el ruido inherente de los retornos financieros. Estos targets presentan estructura, magnitud económica y mejor interpretabilidad que los enfoques basados en regresión directa, alineándose con las recomendaciones del libro.

2. Selección de features

   Se construyó un conjunto reducido y robusto de variables (`roc_30`, `roc_60`, `stoch_k_30`, `atr_norm_10`, `regime_id`) basado en señal predictiva, consistencia out-of-sample y baja redundancia. Esto permite un balance adecuado entre capacidad explicativa y control del overfitting.

3. Split temporal

   La partición en train, validation y test se realizó respetando el orden cronológico por días, evitando completamente el data leakage y garantizando una evaluación realista del modelo en datos futuros.

4. Escalamiento

   El escalado fue aplicado correctamente sobre las features continuas, ajustando el scaler únicamente con el conjunto de entrenamiento y manteniendo intactos tanto el target como las variables categóricas. Se verificó que no se introdujo contaminación entre splits y que la estructura temporal se preserva completamente.

5. Generación de ventanas

   Se implementó un esquema seq2one consistente, donde cada ventana utiliza únicamente información pasada y el target corresponde a la última observación de la secuencia.
   La generación se realizó por día, evitando mezcla entre sesiones y respetando la causalidad temporal.
   Se construyeron correctamente todas las combinaciones de ventanas para ambos targets y múltiples tamaños (`L = 30, 60, 90, 120, 180`).

6. Integridad del pipeline

   Se validó que:

    * no existen NaNs en las ventanas finales,
    * las dimensiones de entrada y salida son correctas,
    * los dtypes son consistentes (`X: float32`, `y: int8`),
    * los datasets se encuentran ordenados temporalmente,
    * y todos los artefactos (datasets, scaler, ventanas, manifest) son reproducibles.

7. Trazabilidad y reproducibilidad

   Se implementó un sistema de persistencia completo que incluye:

    * datasets escalados,
    * ventanas en formato `.npz`,
    * metadata por split,
    * y un manifest global que documenta todas las configuraciones.

    Esto permite reconstruir cualquier experimento de forma determinista.

8. Alineación con el enfoque del libro

   El pipeline sigue las buenas prácticas de *Machine Learning for Trading*:

    * separación estricta entre train y evaluación,
    * control explícito del leakage,
    * uso de targets más robustos que retornos crudos,
    * y preparación adecuada de datos para modelado secuencial.

---

**Conclusión final**

El dataset final y su representación en ventanas están correctamente construidos, libres de leakage y listos para ser utilizados en el entrenamiento de modelos.

El pipeline garantiza consistencia, robustez y reproducibilidad, por lo que el siguiente paso —modelado— puede abordarse con una base sólida y bien validada.
